# MedGemma Pseudo-Report Augmented GNN Inference

This notebook evaluates trained GNN checkpoints without retraining. It generates pseudo radiology reports with `google/medgemma-1.5-4b-it`, encodes those reports with BiomedCLIP, retrieves top-k training studies using a fused image+report embedding, builds a top-k query subgraph, and runs inference with the already trained transformer-GNN and non-transformer-GNN models.

Outputs are saved under `/data/liangz2/openi/multi_kg/medgemma_pseudoreport_gnn_inference`.

In [1]:
INSTALL_DEPS = False
if INSTALL_DEPS:
    import subprocess, sys
    packages = [
        "transformers>=4.50.0", "accelerate", "open_clip_torch", "torch_geometric",
        "faiss-cpu", "pandas", "numpy", "pillow", "tqdm", "scikit-learn",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *packages])

In [2]:
import copy, json, os, random, re
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import average_precision_score, f1_score, roc_auc_score

try:
    from torch_geometric.data import HeteroData
    from torch_geometric.nn import HeteroConv, SAGEConv, GCNConv, GATConv, HGTConv, HANConv, TransformerConv
except ImportError as exc:
    raise ImportError("Install torch_geometric before running GNN inference.") from exc

ImageFile.LOAD_TRUNCATED_IMAGES = True
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

DATA_ROOT = Path("/data/liangz2/openi/multi_kg") if Path("/data/liangz2/openi/multi_kg").exists() else Path.cwd()
KG_10FOLD_DIR = DATA_ROOT / "kg_10fold"
OUTPUT_ROOT = DATA_ROOT / "medgemma_pseudoreport_gnn_inference"
PSEUDO_REPORT_DIR = OUTPUT_ROOT / "pseudo_reports"
BIOMEDCLIP_CACHE_DIR = OUTPUT_ROOT / "biomedclip_report_embeddings"
RESULT_ROOT = OUTPUT_ROOT / "gnn_inference_results"
for p in [OUTPUT_ROOT, PSEUDO_REPORT_DIR, BIOMEDCLIP_CACHE_DIR, RESULT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

SPLIT_CSV_CANDIDATES = [
    DATA_ROOT / "multi_kg_image_json_10fold.csv",
    Path("/data/liangz2/openi/inverse_diff/multi_kg_image_json_10fold.csv"),
    Path("/Users/liangz2/Documents/inverse_diffusion/multi_kg_image_json_10fold.csv"),
]
SPLIT_CSV = next((p for p in SPLIT_CSV_CANDIDATES if p.exists()), SPLIT_CSV_CANDIDATES[0])

MEDGEMMA_MODEL_ID = "google/medgemma-1.5-4b-it"
MEDGEMMA_CACHE_DIR = OUTPUT_ROOT / "hf_cache" / "medgemma-1.5-4b-it"
MEDGEMMA_TOKEN_PATHS = [
    DATA_ROOT / "hug_token.txt",
    Path("/data/liangz2/openi/inverse_diff/roentgen/hug_token.txt"),
    Path.home() / ".cache" / "huggingface" / "token",
]
PSEUDO_REPORT_CSV = PSEUDO_REPORT_DIR / "medgemma_1_5_4b_it_pseudo_reports.csv"
PSEUDO_REPORT_PROGRESS_CSV = PSEUDO_REPORT_DIR / "medgemma_1_5_4b_it_pseudo_reports_progress.csv"
PSEUDO_REPORT_TEXT_EMB_PATH = BIOMEDCLIP_CACHE_DIR / "medgemma_pseudo_report_biomedclip_text_embeddings.npy"
PSEUDO_REPORT_TEXT_EMB_META_PATH = BIOMEDCLIP_CACHE_DIR / "medgemma_pseudo_report_biomedclip_text_embedding_metadata.csv"

BIOMEDCLIP_REPO = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
SCORING_LABELS = (
    "normal",
    "enlarged cardiomediastinum", "cardiomegaly", "lung opacity", "lung lesion", "edema",
    "consolidation", "pneumonia", "atelectasis", "pneumothorax",
    "pleural effusion", "pleural other", "fracture", "support devices",
)
TARGET_COLUMNS = ["y_" + label.replace(" ", "_") for label in SCORING_LABELS]
FOLD_INDICES = list(range(10))
QUERY_TOP_K_VALUES = [50, 100, 200, 500]
GNN_EXPERIMENTS = [
    {"family": "transformer_gnn", "model_name": "hgtconv"},
    {"family": "transformer_gnn", "model_name": "hanconv"},
    {"family": "transformer_gnn", "model_name": "transformerconv"},
    {"family": "non_transformer_gnn", "model_name": "graphsage"},
    {"family": "non_transformer_gnn", "model_name": "gcn"},
    {"family": "non_transformer_gnn", "model_name": "gat"},
]
SKIP_COMPLETED = True
FORCE_RERUN = []
SAVE_PREDICTIONS = True

GENERATE_PSEUDO_REPORTS = True
MAX_PSEUDO_REPORT_IMAGES = None
MEDGEMMA_MAX_NEW_TOKENS = 256
MEDGEMMA_BATCH_SAVE_EVERY = 25
REPORT_RETRIEVAL_IMAGE_WEIGHT = 0.5
REPORT_RETRIEVAL_TEXT_WEIGHT = 0.5
TEXT_EMBED_BATCH_SIZE = 64
TEST_QUERY_BATCH_SIZE = 512
QUERY_INFERENCE_CHUNK_SIZE = 256
INCLUDE_BASE_IMAGE_SIMILAR_EDGES = False
INCLUDE_HAS_FINDING_IN_MESSAGE_PASSING = False
DECODER = "dot"
DROPOUT = 0.20

print("DEVICE", DEVICE, "DTYPE", DTYPE)
print("DATA_ROOT", DATA_ROOT)
print("OUTPUT_ROOT", OUTPUT_ROOT)
print("SPLIT_CSV", SPLIT_CSV)

DEVICE cuda DTYPE torch.bfloat16
DATA_ROOT /data/liangz2/openi/multi_kg
OUTPUT_ROOT /data/liangz2/openi/multi_kg/medgemma_pseudoreport_gnn_inference
SPLIT_CSV /data/liangz2/openi/multi_kg/multi_kg_image_json_10fold.csv


## Metrics and Split Table

In [3]:
def torch_load_any(path: Path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def save_json(obj: Dict, path: Path) -> None:
    path.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")

def sigmoid_np(logits: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-logits))

def evaluate_prob_matrix(probs: np.ndarray, targets: np.ndarray, thresholds: Optional[np.ndarray] = None) -> pd.DataFrame:
    y = targets.astype(int)
    if thresholds is None:
        thresholds = np.full((probs.shape[1],), 0.5, dtype=np.float32)
    rows = []
    for j, label in enumerate(SCORING_LABELS):
        if len(np.unique(y[:, j])) < 2:
            continue
        pred = (probs[:, j] >= thresholds[j]).astype(int)
        rows.append({
            "label": label,
            "auroc": roc_auc_score(y[:, j], probs[:, j]),
            "auprc": average_precision_score(y[:, j], probs[:, j]),
            "f1": f1_score(y[:, j], pred, zero_division=0),
            "threshold": float(thresholds[j]),
            "support": int(y[:, j].sum()),
        })
    return pd.DataFrame(rows)

def metric_summary(metrics: pd.DataFrame, fold: int, method: str, extra: Optional[Dict] = None) -> Dict:
    disease = metrics[metrics["label"] != "normal"] if not metrics.empty else metrics
    row = {
        "fold": int(fold), "method": method,
        "disease_macro_auroc": float(disease["auroc"].mean()) if len(disease) else float("nan"),
        "disease_macro_auprc": float(disease["auprc"].mean()) if len(disease) else float("nan"),
        "disease_macro_f1": float(disease["f1"].mean()) if len(disease) else float("nan"),
        "all_macro_auroc": float(metrics["auroc"].mean()) if len(metrics) else float("nan"),
        "all_macro_auprc": float(metrics["auprc"].mean()) if len(metrics) else float("nan"),
        "all_macro_f1": float(metrics["f1"].mean()) if len(metrics) else float("nan"),
        "num_scored_labels": int(len(metrics)),
    }
    if extra:
        row.update(extra)
    return row

def resolve_image_path(path_value: str, row: Optional[pd.Series] = None) -> str:
    raw = Path(str(path_value)); candidates = [raw]; raw_str = str(raw)
    for old, new in [("/vf/users/liangz2/openi/multi_kg", "/data/liangz2/openi/multi_kg"), ("/data/liangz2/openi/multi_kg", "/vf/users/liangz2/openi/multi_kg")]:
        if raw_str.startswith(old): candidates.append(Path(raw_str.replace(old, new, 1)))
    if row is not None:
        image_name = str(row.get("image_name", Path(raw_str).name)); original_split = str(row.get("original_split", "")).strip()
        if original_split: candidates.append(DATA_ROOT / original_split / image_name)
        candidates += [DATA_ROOT / "train" / image_name, DATA_ROOT / "test" / image_name]
    for c in candidates:
        if c.exists(): return str(c)
    return str(candidates[0])

def load_split_table() -> pd.DataFrame:
    if not SPLIT_CSV.exists(): raise FileNotFoundError(f"Split CSV not found: {SPLIT_CSV_CANDIDATES}")
    frame = pd.read_csv(SPLIT_CSV)
    required = ["study_id", "image_path"] + TARGET_COLUMNS + [f"fold_{i}" for i in FOLD_INDICES]
    missing = [c for c in required if c not in frame.columns]
    if missing: raise ValueError(f"Split CSV missing columns: {missing}")
    frame = frame.copy(); frame["study_id"] = frame["study_id"].astype(str)
    if "subject_id" in frame.columns: frame["subject_id"] = frame["subject_id"].astype(str)
    if "image_name" not in frame.columns: frame["image_name"] = frame["image_path"].map(lambda x: Path(str(x)).name)
    frame["image_path"] = frame["image_path"].astype(str)
    frame["resolved_image_path"] = [resolve_image_path(row["image_path"], row) for _, row in frame.iterrows()]
    frame["row_id"] = np.arange(len(frame), dtype=int)
    frame["image_key"] = frame["study_id"].astype(str) + "::" + frame["image_name"].astype(str)
    return frame

split_table_df = load_split_table()
print("split rows", len(split_table_df))
print(split_table_df[["row_id", "study_id", "image_name", "resolved_image_path"]].head())

split rows 44100
   row_id  study_id    image_name  \
0       0  50003542  50003542.jpg   
1       1  50003755  50003755.jpg   
2       2  50008255  50008255.jpg   
3       3  50010166  50010166.jpg   
4       4  50010229  50010229.jpg   

                                 resolved_image_path  
0  /vf/users/liangz2/openi/multi_kg/train/5000354...  
1  /vf/users/liangz2/openi/multi_kg/train/5000375...  
2  /vf/users/liangz2/openi/multi_kg/train/5000825...  
3  /vf/users/liangz2/openi/multi_kg/train/5001016...  
4  /vf/users/liangz2/openi/multi_kg/train/5001022...  


## Generate MedGemma Pseudo Reports

In [4]:
medgemma_model = None
medgemma_processor = None

def read_hf_token(paths: Sequence[Path]) -> Optional[str]:
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if token: return token.strip()
    for p in paths:
        if p.exists():
            value = p.read_text().strip()
            if value: return value
    return None

def get_medgemma_model():
    global medgemma_model, medgemma_processor
    if medgemma_model is not None: return medgemma_model, medgemma_processor
    token = read_hf_token(MEDGEMMA_TOKEN_PATHS)
    try:
        from transformers import AutoProcessor, AutoModelForImageTextToText
        model_cls = AutoModelForImageTextToText
    except ImportError:
        from transformers import AutoProcessor, AutoModelForMultimodalLM
        model_cls = AutoModelForMultimodalLM
    kwargs = {"torch_dtype": DTYPE, "cache_dir": str(MEDGEMMA_CACHE_DIR)}
    if DEVICE == "cuda": kwargs["device_map"] = "auto"
    if token: kwargs["token"] = token
    medgemma_model = model_cls.from_pretrained(MEDGEMMA_MODEL_ID, **kwargs)
    medgemma_processor = AutoProcessor.from_pretrained(MEDGEMMA_MODEL_ID, token=token, cache_dir=str(MEDGEMMA_CACHE_DIR))
    medgemma_model.eval()
    return medgemma_model, medgemma_processor

PSEUDO_REPORT_PROMPT = """Generate a concise radiology-style report for this frontal chest radiograph.
Use exactly two sections: FINDINGS and IMPRESSION.
Mention visible cardiomegaly, edema, lung opacity, consolidation, pneumonia, atelectasis, pneumothorax, pleural effusion, fracture, and support devices when present.
If the image appears normal, say no acute cardiopulmonary abnormality.
Do not mention uncertainty about being an AI model.
"""

@torch.inference_mode()
def generate_medgemma_report(image_path: str, prompt: str = PSEUDO_REPORT_PROMPT) -> str:
    model, processor = get_medgemma_model()
    image = Image.open(image_path).convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to(model.device, dtype=DTYPE)
    input_len = inputs["input_ids"].shape[-1]
    generation = model.generate(**inputs, max_new_tokens=MEDGEMMA_MAX_NEW_TOKENS, do_sample=False)
    return processor.decode(generation[0][input_len:], skip_special_tokens=True).strip()

def empty_pseudo_report_frame() -> pd.DataFrame:
    return pd.DataFrame(columns=["row_id", "study_id", "subject_id", "image_name", "image_path", "resolved_image_path", "pseudo_report", "status", "error"])

def load_existing_pseudo_reports() -> pd.DataFrame:
    paths = [p for p in [PSEUDO_REPORT_CSV, PSEUDO_REPORT_PROGRESS_CSV] if p.exists()]
    if not paths:
        return empty_pseudo_report_frame()
    # Merge final/progress files and keep the newest copy of each row_id. This makes interrupted runs resumable.
    paths = sorted(paths, key=lambda p: p.stat().st_mtime)
    frames = []
    for path in paths:
        try:
            frames.append(pd.read_csv(path))
            print("Loaded pseudo-report checkpoint:", path)
        except Exception as exc:
            print("Skipping unreadable pseudo-report checkpoint:", path, repr(exc))
    if not frames:
        return empty_pseudo_report_frame()
    out = pd.concat(frames, ignore_index=True, sort=False)
    if "row_id" not in out.columns:
        return empty_pseudo_report_frame()
    out["row_id"] = pd.to_numeric(out["row_id"], errors="coerce")
    out = out.dropna(subset=["row_id"]).copy()
    out["row_id"] = out["row_id"].astype(int)
    return out.drop_duplicates("row_id", keep="last").sort_values("row_id").reset_index(drop=True)

def generate_all_pseudo_reports(frame: pd.DataFrame) -> pd.DataFrame:
    existing = load_existing_pseudo_reports()
    if len(existing) and "status" in existing.columns:
        done_ids = set(existing.loc[existing["status"].astype(str).str.lower().eq("ok"), "row_id"].astype(int).tolist())
    else:
        done_ids = set()
    rows = existing.to_dict(orient="records") if len(existing) else []
    work = frame.head(int(MAX_PSEUDO_REPORT_IMAGES)).copy() if MAX_PSEUDO_REPORT_IMAGES is not None else frame.copy()
    pending = work[~work["row_id"].isin(done_ids)].reset_index(drop=True)
    print("pseudo report existing ok", len(done_ids), "pending", len(pending))
    new_count = 0
    for _, row in tqdm(pending.iterrows(), total=len(pending), desc="MedGemma pseudo reports"):
        out = {"row_id": int(row["row_id"]), "study_id": str(row["study_id"]), "subject_id": str(row.get("subject_id", "")), "image_name": str(row.get("image_name", "")), "image_path": str(row.get("image_path", "")), "resolved_image_path": str(row.get("resolved_image_path", ""))}
        try:
            out["pseudo_report"] = generate_medgemma_report(out["resolved_image_path"]); out["status"] = "ok"; out["error"] = ""
        except Exception as exc:
            out["pseudo_report"] = ""; out["status"] = "error"; out["error"] = repr(exc)
        rows.append(out)
        new_count += 1
        if new_count % MEDGEMMA_BATCH_SAVE_EVERY == 0:
            pd.DataFrame(rows).drop_duplicates("row_id", keep="last").sort_values("row_id").to_csv(PSEUDO_REPORT_PROGRESS_CSV, index=False)
    result = pd.DataFrame(rows).drop_duplicates("row_id", keep="last").sort_values("row_id")
    result.to_csv(PSEUDO_REPORT_CSV, index=False); result.to_csv(PSEUDO_REPORT_PROGRESS_CSV, index=False)
    return result

# Optional one-image sanity check before full generation:
# print(generate_medgemma_report(split_table_df.iloc[0]["resolved_image_path"]))

pseudo_reports_df = generate_all_pseudo_reports(split_table_df) if GENERATE_PSEUDO_REPORTS else load_existing_pseudo_reports()
print("pseudo reports", pseudo_reports_df.shape)

Loaded pseudo-report checkpoint: /data/liangz2/openi/multi_kg/medgemma_pseudoreport_gnn_inference/pseudo_reports/medgemma_1_5_4b_it_pseudo_reports.csv
Loaded pseudo-report checkpoint: /data/liangz2/openi/multi_kg/medgemma_pseudoreport_gnn_inference/pseudo_reports/medgemma_1_5_4b_it_pseudo_reports_progress.csv
pseudo report existing ok 44100 pending 0


MedGemma pseudo reports: 0it [00:00, ?it/s]

pseudo reports (44100, 9)


## Encode Pseudo Reports with BiomedCLIP

In [5]:
import hashlib

biomedclip_model = None
biomedclip_tokenizer = None

def get_biomedclip_model():
    global biomedclip_model, biomedclip_tokenizer
    if biomedclip_model is None:
        from open_clip import create_model_from_pretrained, get_tokenizer
        model, _ = create_model_from_pretrained(BIOMEDCLIP_REPO)
        model = model.to(DEVICE); model.eval()
        for p in model.parameters(): p.requires_grad = False
        biomedclip_model = model; biomedclip_tokenizer = get_tokenizer(BIOMEDCLIP_REPO)
    return biomedclip_model, biomedclip_tokenizer

@torch.inference_mode()
def encode_texts_biomedclip(texts: Sequence[str], batch_size: int = TEXT_EMBED_BATCH_SIZE) -> np.ndarray:
    model, tokenizer = get_biomedclip_model(); chunks=[]
    for start in tqdm(range(0, len(texts), batch_size), desc="BiomedCLIP report text embeddings"):
        tokens = tokenizer(list(texts[start:start+batch_size])).to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            feats = model.encode_text(tokens)
        feats = F.normalize(feats.float(), dim=-1)
        chunks.append(feats.cpu().numpy().astype("float32"))
    return np.concatenate(chunks, axis=0) if chunks else np.empty((0,512), dtype="float32")

def pseudo_report_cache_signature(frame: pd.DataFrame, pseudo_reports: pd.DataFrame) -> Tuple[str, int]:
    report_map = pseudo_reports.drop_duplicates("row_id", keep="last").set_index("row_id") if len(pseudo_reports) else pd.DataFrame()
    hasher = hashlib.sha256()
    ok_count = 0
    for _, row in frame.iterrows():
        rid = int(row["row_id"])
        text = ""
        status = ""
        if len(report_map) and rid in report_map.index:
            text = str(report_map.loc[rid].get("pseudo_report", "") or "")
            status = str(report_map.loc[rid].get("status", "") or "")
        if status.lower() == "ok":
            ok_count += 1
        hasher.update(str(rid).encode("utf-8")); hasher.update(b"\0")
        hasher.update(status.encode("utf-8", errors="ignore")); hasher.update(b"\0")
        hasher.update(text.encode("utf-8", errors="ignore")); hasher.update(b"\0")
    return hasher.hexdigest(), ok_count

def get_or_create_pseudo_report_text_embeddings(frame: pd.DataFrame, pseudo_reports: pd.DataFrame) -> np.ndarray:
    signature, ok_count = pseudo_report_cache_signature(frame, pseudo_reports)
    if PSEUDO_REPORT_TEXT_EMB_PATH.exists() and PSEUDO_REPORT_TEXT_EMB_META_PATH.exists():
        meta = pd.read_csv(PSEUDO_REPORT_TEXT_EMB_META_PATH)
        cached_signature = str(meta.get("pseudo_report_signature", pd.Series([""])).iloc[0]) if len(meta) else ""
        if len(meta) == len(frame) and np.array_equal(meta["row_id"].to_numpy(), frame["row_id"].to_numpy()) and cached_signature == signature:
            print("Loading cached pseudo-report text embeddings", PSEUDO_REPORT_TEXT_EMB_PATH)
            return np.load(PSEUDO_REPORT_TEXT_EMB_PATH).astype("float32")
    report_map = pseudo_reports.drop_duplicates("row_id", keep="last").set_index("row_id")
    texts=[]
    for _, row in frame.iterrows():
        rid=int(row["row_id"]); text=""
        if rid in report_map.index: text = str(report_map.loc[rid, "pseudo_report"] or "")
        texts.append(text if text.strip() else "Chest radiograph. No pseudo report available.")
    embeddings = encode_texts_biomedclip(texts)
    np.save(PSEUDO_REPORT_TEXT_EMB_PATH, embeddings.astype("float32"))
    meta = frame[["row_id", "study_id", "image_name", "resolved_image_path"]].copy()
    meta["pseudo_report_signature"] = signature
    meta["pseudo_report_ok_count"] = int(ok_count)
    meta["pseudo_report_total_rows"] = int(len(pseudo_reports))
    meta.to_csv(PSEUDO_REPORT_TEXT_EMB_META_PATH, index=False)
    return embeddings.astype("float32")

pseudo_report_text_embeddings = get_or_create_pseudo_report_text_embeddings(split_table_df, pseudo_reports_df)
print("pseudo_report_text_embeddings", pseudo_report_text_embeddings.shape)

Loading cached pseudo-report text embeddings /data/liangz2/openi/multi_kg/medgemma_pseudoreport_gnn_inference/biomedclip_report_embeddings/medgemma_pseudo_report_biomedclip_text_embeddings.npy
pseudo_report_text_embeddings (44100, 512)


## GNN Architectures Matching Saved Checkpoints

In [6]:
def merge_layer_output(prev_z, out_z, norm_dict, dropout, training):
    next_z={}
    for nt,x in prev_z.items():
        y=out_z.get(nt)
        if y is None: next_z[nt]=x; continue
        y=F.relu(y); y=norm_dict[nt](y); y=F.dropout(y, p=dropout, training=training)
        next_z[nt]=x+y if x.shape==y.shape else y
    return next_z

class BaseImageLabelDecoder(nn.Module):
    def _init_decoder(self, hidden_dim, dropout, decoder):
        self.decoder=decoder
        if decoder == "mlp": self.decoder_mlp=nn.Sequential(nn.Linear(2*hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 1))
        elif decoder != "dot": raise ValueError(f"Unknown decoder {decoder}")
    def decode_image_label_logits(self, z_dict, label_indices):
        image_z=z_dict["image"]; label_z=z_dict["label"][torch.as_tensor(label_indices, device=image_z.device)]
        if self.decoder == "dot": return image_z @ label_z.t()
        pair=torch.cat([image_z[:,None,:].expand(-1,len(label_indices),-1), label_z[None,:,:].expand(image_z.size(0),-1,-1)], dim=-1)
        return self.decoder_mlp(pair).squeeze(-1)

class HeteroGraphSAGE(BaseImageLabelDecoder):
    def __init__(self, metadata, input_dims, hidden_dim=256, num_layers=2, heads=1, dropout=0.2, decoder=DECODER):
        super().__init__(); nts, ets=metadata; self.dropout=dropout
        self.proj=nn.ModuleDict({nt:nn.Linear(int(input_dims[nt]), hidden_dim) for nt in nts if nt in input_dims})
        self.convs=nn.ModuleList([HeteroConv({et:SAGEConv((hidden_dim,hidden_dim), hidden_dim) for et in ets}, aggr="sum") for _ in range(num_layers)])
        self.norms=nn.ModuleList([nn.ModuleDict({nt:nn.LayerNorm(hidden_dim) for nt in self.proj}) for _ in range(num_layers)])
        self._init_decoder(hidden_dim, dropout, decoder)
    def forward(self, x_dict, edge_index_dict):
        z={nt:F.relu(self.proj[nt](x.float())) for nt,x in x_dict.items() if nt in self.proj}
        for conv,norm in zip(self.convs,self.norms): z=merge_layer_output(z, conv(z, edge_index_dict), norm, self.dropout, self.training)
        return z

def make_hgt_conv(hidden_dim, metadata, heads):
    try: return HGTConv(hidden_dim, hidden_dim, metadata, heads=heads, group="sum")
    except TypeError as exc:
        if "group" in str(exc): return HGTConv(hidden_dim, hidden_dim, metadata, heads=heads)
        raise

class HeteroHGT(BaseImageLabelDecoder):
    def __init__(self, metadata, input_dims, hidden_dim=256, num_layers=2, heads=4, dropout=0.2, decoder=DECODER):
        super().__init__(); nts,_=metadata; self.dropout=dropout
        self.proj=nn.ModuleDict({nt:nn.Linear(int(input_dims[nt]), hidden_dim) for nt in nts if nt in input_dims})
        self.convs=nn.ModuleList([make_hgt_conv(hidden_dim, metadata, heads) for _ in range(num_layers)])
        self.norms=nn.ModuleList([nn.ModuleDict({nt:nn.LayerNorm(hidden_dim) for nt in self.proj}) for _ in range(num_layers)])
        self._init_decoder(hidden_dim, dropout, decoder)
    def forward(self, x_dict, edge_index_dict):
        z={nt:F.relu(self.proj[nt](x.float())) for nt,x in x_dict.items() if nt in self.proj}
        for conv,norm in zip(self.convs,self.norms): z=merge_layer_output(z, conv(z, edge_index_dict), norm, self.dropout, self.training)
        return z

class HeteroHAN(HeteroHGT):
    def __init__(self, metadata, input_dims, hidden_dim=256, num_layers=2, heads=4, dropout=0.2, decoder=DECODER):
        BaseImageLabelDecoder.__init__(self); nts,_=metadata; self.dropout=dropout
        self.proj=nn.ModuleDict({nt:nn.Linear(int(input_dims[nt]), hidden_dim) for nt in nts if nt in input_dims})
        in_channels={nt:hidden_dim for nt in nts if nt in input_dims}
        self.convs=nn.ModuleList([HANConv(in_channels, hidden_dim, metadata=metadata, heads=heads, dropout=dropout) for _ in range(num_layers)])
        self.norms=nn.ModuleList([nn.ModuleDict({nt:nn.LayerNorm(hidden_dim) for nt in self.proj}) for _ in range(num_layers)])
        self._init_decoder(hidden_dim, dropout, decoder)

class HeteroTransformerConv(HeteroGraphSAGE):
    def __init__(self, metadata, input_dims, hidden_dim=128, num_layers=1, heads=2, dropout=0.2, decoder=DECODER):
        BaseImageLabelDecoder.__init__(self); nts,ets=metadata; self.dropout=dropout
        self.proj=nn.ModuleDict({nt:nn.Linear(int(input_dims[nt]), hidden_dim) for nt in nts if nt in input_dims})
        self.convs=nn.ModuleList([HeteroConv({et:TransformerConv((hidden_dim,hidden_dim), hidden_dim, heads=heads, concat=False, dropout=dropout) for et in ets}, aggr="sum") for _ in range(num_layers)])
        self.norms=nn.ModuleList([nn.ModuleDict({nt:nn.LayerNorm(hidden_dim) for nt in self.proj}) for _ in range(num_layers)])
        self._init_decoder(hidden_dim, dropout, decoder)

class HeteroGAT(HeteroGraphSAGE):
    def __init__(self, metadata, input_dims, hidden_dim=128, num_layers=2, heads=2, dropout=0.2, decoder=DECODER):
        BaseImageLabelDecoder.__init__(self); nts,ets=metadata; self.dropout=dropout
        self.proj=nn.ModuleDict({nt:nn.Linear(int(input_dims[nt]), hidden_dim) for nt in nts if nt in input_dims})
        self.convs=nn.ModuleList([HeteroConv({et:GATConv((hidden_dim,hidden_dim), hidden_dim, heads=heads, concat=False, dropout=dropout, add_self_loops=False) for et in ets}, aggr="sum") for _ in range(num_layers)])
        self.norms=nn.ModuleList([nn.ModuleDict({nt:nn.LayerNorm(hidden_dim) for nt in self.proj}) for _ in range(num_layers)])
        self._init_decoder(hidden_dim, dropout, decoder)

class HeteroGCN(BaseImageLabelDecoder):
    def __init__(self, metadata, input_dims, hidden_dim=256, num_layers=2, heads=1, dropout=0.2, decoder=DECODER):
        super().__init__(); nts,_=metadata; self.node_types=[nt for nt in nts if nt in input_dims]; self.dropout=dropout
        self.proj=nn.ModuleDict({nt:nn.Linear(int(input_dims[nt]), hidden_dim) for nt in self.node_types})
        self.convs=nn.ModuleList([GCNConv(hidden_dim, hidden_dim, add_self_loops=True, normalize=True) for _ in range(num_layers)])
        self.norm=nn.LayerNorm(hidden_dim); self._init_decoder(hidden_dim, dropout, decoder)
    def flatten_graph(self,z,edge_index_dict):
        offsets={}; chunks=[]; start=0
        for nt in self.node_types:
            x=z[nt]; end=start+x.size(0); offsets[nt]=(start,end); chunks.append(x); start=end
        hx=torch.cat(chunks,dim=0); edges=[]
        for (st,_,dt),ei in edge_index_dict.items():
            if st not in offsets or dt not in offsets: continue
            e=ei.clone().to(hx.device); e[0]+=offsets[st][0]; e[1]+=offsets[dt][0]; edges.append(e)
        he=torch.cat(edges,dim=1) if edges else torch.empty((2,0), dtype=torch.long, device=hx.device)
        return hx, he, offsets
    def forward(self,x_dict,edge_index_dict):
        z={nt:F.relu(self.proj[nt](x.float())) for nt,x in x_dict.items() if nt in self.proj}
        h,ei,offsets=self.flatten_graph(z,edge_index_dict)
        for conv in self.convs:
            y=conv(h,ei); y=F.relu(y); y=self.norm(y); y=F.dropout(y,p=self.dropout,training=self.training); h=h+y if h.shape==y.shape else y
        return {nt:h[s:e] for nt,(s,e) in offsets.items()}

def build_model(model_name, metadata, input_dims, checkpoint):
    cfg={"hidden_dim":int(checkpoint.get("hidden_dim",256)), "num_layers":int(checkpoint.get("num_layers",2)), "heads":int(checkpoint.get("num_heads",1)), "dropout":float(checkpoint.get("dropout",DROPOUT)), "decoder":checkpoint.get("decoder",DECODER)}
    if model_name=="graphsage": return HeteroGraphSAGE(metadata,input_dims,**cfg)
    if model_name=="hgtconv": return HeteroHGT(metadata,input_dims,**cfg)
    if model_name=="hanconv": return HeteroHAN(metadata,input_dims,**cfg)
    if model_name=="transformerconv": return HeteroTransformerConv(metadata,input_dims,**cfg)
    if model_name=="gcn": return HeteroGCN(metadata,input_dims,**cfg)
    if model_name=="gat": return HeteroGAT(metadata,input_dims,**cfg)
    raise ValueError(model_name)

## KG Loading, Pseudo-Report Retrieval, and Query Subgraph

In [7]:
def dict_to_heterodata(obj):
    data=HeteroData()
    for nt,x in obj.get("x_dict",{}).items():
        data[nt].x=x.float()
        if nt in obj.get("node_ids",{}): data[nt].node_id=list(obj["node_ids"][nt])
        if nt in obj.get("y_dict",{}): data[nt].y=obj["y_dict"][nt].float()
    for et,ei in obj.get("edge_index_dict",{}).items():
        data[et].edge_index=ei.long()
        if et in obj.get("edge_weight_dict",{}): data[et].edge_weight=obj["edge_weight_dict"][et].float()
    return data

def load_fold_data(fold):
    fd=KG_10FOLD_DIR/f"fold{fold}"; obj=torch_load_any(fd/f"kg_heterodata_fold{fold}.pt", map_location="cpu")
    data=obj if isinstance(obj,HeteroData) else dict_to_heterodata(obj)
    return data, pd.read_csv(fd/f"train_metadata_fold{fold}.csv"), pd.read_csv(fd/f"test_query_metadata_fold{fold}.csv"), np.load(fd/f"test_query_embeddings_fold{fold}.npy").astype("float32")

def label_names_from_data(data):
    if hasattr(data["label"],"label_name"): return [str(x) for x in data["label"].label_name]
    if hasattr(data["label"],"node_id"):
        return [str(x).split("label::",1)[-1] if str(x).startswith("label::") else str(x) for x in data["label"].node_id]
    raise RuntimeError("Label metadata missing")

def label_indices_for_scoring(data):
    lookup={name:i for i,name in enumerate(label_names_from_data(data))}
    return [lookup[label] for label in SCORING_LABELS]

def message_passing_edge_index_dict(data, include_has_finding=INCLUDE_HAS_FINDING_IN_MESSAGE_PASSING):
    out={}
    for et in data.edge_types:
        if (not include_has_finding) and et[1] in {"has_finding","rev_has_finding"}: continue
        if hasattr(data[et],"edge_index"): out[et]=data[et].edge_index
    return out

def input_dims_from_data(data): return {nt:int(data[nt].x.size(-1)) for nt in data.node_types if hasattr(data[nt],"x")}
def target_matrix_from_metadata(df): return df[TARGET_COLUMNS].to_numpy(dtype=np.float32)
def l2_normalize_np(x, eps=1e-12): return (x/np.maximum(np.linalg.norm(x,axis=1,keepdims=True),eps)).astype("float32")
def combine_image_report_embeddings(img, txt): return l2_normalize_np(REPORT_RETRIEVAL_IMAGE_WEIGHT*l2_normalize_np(img)+REPORT_RETRIEVAL_TEXT_WEIGHT*l2_normalize_np(txt))

def frame_row_ids(frame):
    local=frame.copy()
    if "image_name" not in local.columns: local["image_name"]=local["image_path"].map(lambda x: Path(str(x)).name)
    local["image_key"]=local["study_id"].astype(str)+"::"+local["image_name"].astype(str)
    lookup=split_table_df.set_index("image_key")["row_id"].to_dict(); ids=local["image_key"].map(lookup)
    if ids.isna().any(): raise RuntimeError(f"Could not map {ids.isna().sum()} fold rows to split table")
    return ids.to_numpy(dtype=int)

def topk_inner_product(train_embeddings, query_embeddings, top_k, batch_size=TEST_QUERY_BATCH_SIZE):
    train=l2_normalize_np(train_embeddings); query=l2_normalize_np(query_embeddings); all_sims=[]; all_idx=[]
    try:
        import faiss
        index=faiss.IndexFlatIP(train.shape[1]); index.add(np.ascontiguousarray(train))
        for s in tqdm(range(0,len(query),batch_size), desc=f"pseudo-report FAISS top_k={top_k}"):
            sims,idx=index.search(np.ascontiguousarray(query[s:s+batch_size]), min(top_k,len(train))); all_sims.append(sims.astype("float32")); all_idx.append(idx.astype("int64"))
    except ImportError:
        for s in tqdm(range(0,len(query),batch_size), desc=f"dense retrieval top_k={top_k}"):
            scores=query[s:s+batch_size]@train.T; kth=min(top_k,scores.shape[1])-1; idx=np.argpartition(-scores,kth=kth,axis=1)[:,:top_k]
            order=np.take_along_axis(scores,idx,axis=1).argsort(axis=1)[:,::-1]; idx=np.take_along_axis(idx,order,axis=1); sims=np.take_along_axis(scores,idx,axis=1)
            all_sims.append(sims.astype("float32")); all_idx.append(idx.astype("int64"))
    return np.concatenate(all_sims), np.concatenate(all_idx)

def append_edge_index(data, edge_type, edge_index_new, edge_weight_new=None):
    store=data[edge_type]
    store.edge_index=torch.cat([store.edge_index.cpu(), edge_index_new.cpu()], dim=1) if hasattr(store,"edge_index") else edge_index_new.cpu()
    if edge_weight_new is not None: store.edge_weight=torch.cat([store.edge_weight.cpu(), edge_weight_new.cpu()], dim=0) if hasattr(store,"edge_weight") else edge_weight_new.cpu()

def safe_node_attr_values(store, attr, old_indices, node_type):
    if not hasattr(store, attr):
        return None
    vals=list(getattr(store, attr))
    copied=[]
    for old_i in old_indices.tolist():
        old_i=int(old_i)
        if 0 <= old_i < len(vals):
            copied.append(vals[old_i])
        else:
            copied.append(f"{node_type}::{attr}::generated_{old_i}")
    return copied

def build_query_subgraph(data, query_embeddings, query_targets, neighbor_idx, neighbor_sims):
    base=copy.deepcopy(data).cpu(); train_count=int(base["image"].x.size(0)); nq=int(query_embeddings.shape[0])
    base["image"].x=torch.cat([base["image"].x.float(), torch.from_numpy(query_embeddings).float()], dim=0)
    base["image"].y=torch.cat([base["image"].y.float(), torch.from_numpy(query_targets).float()], dim=0) if hasattr(base["image"],"y") else torch.from_numpy(query_targets).float()
    src=[]; dst=[]; weights=[]
    for qi in range(neighbor_idx.shape[0]):
        q=train_count+qi
        for r in range(neighbor_idx.shape[1]): src.append(q); dst.append(int(neighbor_idx[qi,r])); weights.append(float(neighbor_sims[qi,r]))
    q_edge=torch.tensor([src,dst], dtype=torch.long); q_w=torch.tensor(weights, dtype=torch.float32)
    append_edge_index(base,("image","similar_to","image"),q_edge,q_w); append_edge_index(base,("image","rev_similar_to","image"),q_edge.flip(0),q_w)
    keep={nt:torch.zeros(base[nt].x.size(0),dtype=torch.bool) for nt in base.node_types if hasattr(base[nt],"x")}
    selected=np.unique(neighbor_idx.reshape(-1)).astype(int); query_global=list(range(train_count,train_count+nq))
    keep["image"][torch.from_numpy(selected)]=True; keep["image"][torch.tensor(query_global)]=True
    if "label" in keep: keep["label"][:]=True
    for _ in range(2):
        for et in base.edge_types:
            st,rel,dt=et
            if rel in {"has_finding","rev_has_finding"}: continue
            if rel in {"similar_to","rev_similar_to"} and not INCLUDE_BASE_IMAGE_SIMILAR_EDGES: continue
            if st not in keep or dt not in keep or not hasattr(base[et],"edge_index"): continue
            ei=base[et].edge_index.cpu(); src_keep=keep[st][ei[0]]; dst_keep=keep[dt][ei[1]]
            if dt != "image": keep[dt][ei[1][src_keep]]=True
            if st != "image": keep[st][ei[0][dst_keep]]=True
    keep["image"][torch.from_numpy(selected)]=True; keep["image"][torch.tensor(query_global)]=True
    sub=HeteroData(); old_to_new={}
    for nt in base.node_types:
        if nt not in keep or not hasattr(base[nt],"x"): continue
        old=torch.where(keep[nt])[0]; mapper=torch.full((base[nt].x.size(0),),-1,dtype=torch.long); mapper[old]=torch.arange(old.numel()); old_to_new[nt]=mapper
        sub[nt].x=base[nt].x[old].float()
        if hasattr(base[nt],"y"): sub[nt].y=base[nt].y[old].float()
        for attr in ["node_id","label_name","canonical_name"]:
            copied=safe_node_attr_values(base[nt],attr,old,nt)
            if copied is not None: setattr(sub[nt],attr,copied)
    for et in base.edge_types:
        st,rel,dt=et
        if st not in old_to_new or dt not in old_to_new or not hasattr(base[et],"edge_index"): continue
        ei=base[et].edge_index.cpu()
        if rel in {"similar_to","rev_similar_to"} and not INCLUDE_BASE_IMAGE_SIMILAR_EDGES: mask=(ei[0]>=train_count) if rel=="similar_to" else (ei[1]>=train_count)
        else: mask=torch.ones(ei.size(1), dtype=torch.bool)
        mask=mask & keep[st][ei[0]] & keep[dt][ei[1]]; sel=ei[:,mask]
        sub[et].edge_index=torch.stack([old_to_new[st][sel[0]], old_to_new[dt][sel[1]]], dim=0) if sel.numel() else torch.empty((2,0), dtype=torch.long)
        if hasattr(base[et],"edge_weight"): sub[et].edge_weight=base[et].edge_weight.cpu()[mask].float()
    query_local=[int(old_to_new["image"][i]) for i in query_global]
    return sub, query_local, label_indices_for_scoring(sub)

## Load Trained Checkpoints and Run Inference

In [8]:
def checkpoint_path(family, model_name, top_k, fold):
    root = DATA_ROOT / ("kg_transformer_gnn_10fold" if family == "transformer_gnn" else "kg_non_transformer_gnn_10fold")
    return root / model_name / f"top_k_{top_k}" / f"fold{fold}" / "models" / f"hetero_{model_name}_fold{fold}_topk{top_k}.pt"

def threshold_path(family, model_name, top_k, fold):
    root = DATA_ROOT / ("kg_transformer_gnn_10fold" if family == "transformer_gnn" else "kg_non_transformer_gnn_10fold")
    return root / model_name / f"top_k_{top_k}" / f"fold{fold}" / "metrics" / f"{model_name}_val_f1_thresholds.csv"

def load_thresholds(path):
    if path.exists():
        f=pd.read_csv(path); lookup=dict(zip(f["label"].astype(str), f["threshold"].astype(float)))
        return np.array([lookup.get(label,0.5) for label in SCORING_LABELS], dtype=np.float32)
    return np.full((len(SCORING_LABELS),),0.5,dtype=np.float32)

def output_dirs(family, model_name, top_k, fold):
    root=RESULT_ROOT/family/model_name/f"top_k_{top_k}"/f"fold{fold}"; dirs={"root":root,"metrics":root/"metrics","predictions":root/"predictions","neighbors":root/"neighbors"}
    for p in dirs.values(): p.mkdir(parents=True, exist_ok=True)
    return dirs

def run_model_fold_inference(family, model_name, top_k, fold):
    dirs=output_dirs(family,model_name,top_k,fold); done=dirs["root"]/"FOLD_COMPLETE.json"; summary_csv=dirs["metrics"]/f"{model_name}_medgemma_pseudoreport_method_summary.csv"
    if SKIP_COMPLETED and (family,model_name,top_k,fold) not in FORCE_RERUN and done.exists() and summary_csv.exists():
        print(f"{family}/{model_name}/k={top_k}/fold={fold}: complete, skipping"); return pd.read_csv(summary_csv).to_dict(orient="records")
    ckpt_path=checkpoint_path(family,model_name,top_k,fold)
    if not ckpt_path.exists(): print("Missing checkpoint, skipping", ckpt_path); return []
    data, train_df, test_df, test_image_embeddings=load_fold_data(fold)
    train_ids=frame_row_ids(train_df); test_ids=frame_row_ids(test_df)
    train_report_embeddings=pseudo_report_text_embeddings[train_ids]; test_report_embeddings=pseudo_report_text_embeddings[test_ids]
    train_image_embeddings=data["image"].x.cpu().numpy().astype("float32")
    train_ret=combine_image_report_embeddings(train_image_embeddings,train_report_embeddings); test_ret=combine_image_report_embeddings(test_image_embeddings,test_report_embeddings)
    sims,idx=topk_inner_product(train_ret,test_ret,top_k=top_k)
    ckpt=torch_load_any(ckpt_path,map_location="cpu"); full_edges=message_passing_edge_index_dict(data); metadata=(data.node_types,list(full_edges.keys())); input_dims=input_dims_from_data(data)
    model=build_model(model_name,metadata,input_dims,ckpt).to(DEVICE); model.load_state_dict(ckpt["state_dict"]); model.eval()
    test_targets=target_matrix_from_metadata(test_df); chunks=[]
    for s in tqdm(range(0,len(test_df),QUERY_INFERENCE_CHUNK_SIZE), desc=f"{family}/{model_name} fold={fold} top_k={top_k}"):
        e=min(s+QUERY_INFERENCE_CHUNK_SIZE,len(test_df)); sub,q_local,label_indices=build_query_subgraph(data,test_image_embeddings[s:e],test_targets[s:e],idx[s:e],sims[s:e])
        sub_edges=message_passing_edge_index_dict(sub); sub=sub.to(DEVICE); sub_edges={et:ei.to(DEVICE) for et,ei in sub_edges.items()}
        with torch.no_grad():
            z=model(sub.x_dict,sub_edges); logits=model.decode_image_label_logits(z,label_indices); chunk=logits[torch.tensor(q_local,device=logits.device)].cpu().numpy().astype("float32"); chunks.append(sigmoid_np(chunk))
        del sub, sub_edges, z, logits
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    probs=np.concatenate(chunks,axis=0); thresholds=load_thresholds(threshold_path(family,model_name,top_k,fold))
    metrics_fixed=evaluate_prob_matrix(probs,test_targets); metrics_tuned=evaluate_prob_matrix(probs,test_targets,thresholds=thresholds)
    fixed_path=dirs["metrics"]/f"{model_name}_medgemma_pseudoreport_test_metrics.csv"; tuned_path=dirs["metrics"]/f"{model_name}_medgemma_pseudoreport_test_metrics_val_tuned_thresholds.csv"
    metrics_fixed.to_csv(fixed_path,index=False); metrics_tuned.to_csv(tuned_path,index=False)
    if SAVE_PREDICTIONS:
        rows=[]
        for i,row in test_df.reset_index(drop=True).iterrows():
            out={"study_id":str(row.get("study_id","")),"image_path":str(row.get("image_path",""))}
            for j,label in enumerate(SCORING_LABELS): out[f"prob_{label}"]=float(probs[i,j]); out[f"target_{label}"]=int(test_targets[i,j]>=0.5)
            rows.append(out)
        pd.DataFrame(rows).to_csv(dirs["predictions"]/f"{model_name}_medgemma_pseudoreport_test_predictions.csv",index=False)
    common={"family":family,"model_name":model_name,"query_top_k":int(top_k),"retrieval":"biomedclip_image_plus_medgemma_pseudoreport","image_weight":REPORT_RETRIEVAL_IMAGE_WEIGHT,"text_weight":REPORT_RETRIEVAL_TEXT_WEIGHT,"checkpoint_path":str(ckpt_path),"include_has_finding_in_message_passing":INCLUDE_HAS_FINDING_IN_MESSAGE_PASSING,"include_base_image_similar_edges":INCLUDE_BASE_IMAGE_SIMILAR_EDGES}
    s1=metric_summary(metrics_fixed,fold,f"{family}_{model_name}_medgemma_pseudoreport",{**common,"threshold_mode":"fixed_0.5","metrics_path":str(fixed_path)})
    s2=metric_summary(metrics_tuned,fold,f"{family}_{model_name}_medgemma_pseudoreport",{**common,"threshold_mode":"val_tuned_f1","metrics_path":str(tuned_path)})
    pd.DataFrame([s1,s2]).to_csv(summary_csv,index=False); save_json({"fold":fold,"status":"complete","summary_path":str(summary_csv)},done)
    return [s1,s2]

## Run Inference

In [9]:
all_rows=[]
for exp in GNN_EXPERIMENTS:
    family=exp["family"]; model_name=exp["model_name"]
    for top_k in QUERY_TOP_K_VALUES:
        print(f"\n######## {family}/{model_name} top_k={top_k} ########")
        for fold in FOLD_INDICES:
            print(f"\n===== {family}/{model_name} top_k={top_k} fold={fold} =====")
            rows=run_model_fold_inference(family,model_name,top_k,fold)
            all_rows.extend(rows)
            pd.DataFrame(all_rows).to_csv(OUTPUT_ROOT/"medgemma_pseudoreport_gnn_all_results_progress.csv",index=False)
all_results_df=pd.DataFrame(all_rows)
all_results_df.to_csv(OUTPUT_ROOT/"medgemma_pseudoreport_gnn_all_results.csv",index=False)
display(all_results_df)


######## transformer_gnn/hgtconv top_k=50 ########

===== transformer_gnn/hgtconv top_k=50 fold=0 =====
transformer_gnn/hgtconv/k=50/fold=0: complete, skipping

===== transformer_gnn/hgtconv top_k=50 fold=1 =====
transformer_gnn/hgtconv/k=50/fold=1: complete, skipping

===== transformer_gnn/hgtconv top_k=50 fold=2 =====
transformer_gnn/hgtconv/k=50/fold=2: complete, skipping

===== transformer_gnn/hgtconv top_k=50 fold=3 =====
transformer_gnn/hgtconv/k=50/fold=3: complete, skipping

===== transformer_gnn/hgtconv top_k=50 fold=4 =====
transformer_gnn/hgtconv/k=50/fold=4: complete, skipping

===== transformer_gnn/hgtconv top_k=50 fold=5 =====
transformer_gnn/hgtconv/k=50/fold=5: complete, skipping

===== transformer_gnn/hgtconv top_k=50 fold=6 =====
transformer_gnn/hgtconv/k=50/fold=6: complete, skipping

===== transformer_gnn/hgtconv top_k=50 fold=7 =====
transformer_gnn/hgtconv/k=50/fold=7: complete, skipping

===== transformer_gnn/hgtconv top_k=50 fold=8 =====
transformer_gnn/hgtconv

,fold,method,disease_macro_auroc,disease_macro_auprc,disease_macro_f1,all_macro_auroc,all_macro_auprc,all_macro_f1,num_scored_labels,family,model_name,query_top_k,retrieval,image_weight,text_weight,checkpoint_path,include_has_finding_in_message_passing,include_base_image_similar_edges,threshold_mode,metrics_path
0,0,transformer_gnn_hgtconv_medgemma_pseudoreport,0.703436,0.107073,0.109045,0.687148,0.143019,0.157865,14,transformer_gnn,hgtconv,50,biomedclip_image_plus_medgemma_pseudoreport,0.5,0.5,/data/liangz2/openi/multi_kg/kg_transformer_gn...,False,False,fixed_0.5,/data/liangz2/openi/multi_kg/medgemma_pseudore...
1,0,transformer_gnn_hgtconv_medgemma_pseudoreport,0.703436,0.107073,0.030093,0.687148,0.143019,0.084564,14,transformer_gnn,hgtconv,50,biomedclip_image_plus_medgemma_pseudoreport,0.5,0.5,/data/liangz2/openi/multi_kg/kg_transformer_gn...,False,False,val_tuned_f1,/data/liangz2/openi/multi_kg/medgemma_pseudore...
2,1,transformer_gnn_hgtconv_medgemma_pseudoreport,0.768565,0.145843,0.067286,0.732192,0.174239,0.118965,14,transformer_gnn,hgtconv,50,biomedclip_image_plus_medgemma_pseudoreport,0.5,0.5,/data/liangz2/openi/multi_kg/kg_transformer_gn...,False,False,fixed_0.5,/data/liangz2/openi/multi_kg/medgemma_pseudore...
3,1,transformer_gnn_hgtconv_medgemma_pseudoreport,0.768565,0.145843,0.013015,0.732192,0.174239,0.068570,14,transformer_gnn,hgtconv,50,biomedclip_image_plus_medgemma_pseudoreport,0.5,0.5,/data/liangz2/openi/multi_kg/kg_transformer_gn...,False,False,val_tuned_f1,/data/liangz2/openi/multi_kg/medgemma_pseudore...
4,2,transformer_gnn_hgtconv_medgemma_pseudoreport,0.756363,0.135834,0.092251,0.740163,0.173822,0.142146,14,transformer_gnn,hgtconv,50,biomedclip_image_plus_medgemma_pseudoreport,0.5,0.5,/data/liangz2/openi/multi_kg/kg_transformer_gn...,False,False,fixed_0.5,/data/liangz2/openi/multi_kg/medgemma_pseudore...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
475,7,non_transformer_gnn_gat_medgemma_pseudoreport,0.697444,0.141059,0.163742,0.700820,0.189081,0.208519,14,non_transformer_gnn,gat,500,biomedclip_image_plus_medgemma_pseudoreport,0.5,0.5,/data/liangz2/openi/multi_kg/kg_non_transforme...,False,False,val_tuned_f1,/data/liangz2/openi/multi_kg/medgemma_pseudore...
476,8,non_transformer_gnn_gat_medgemma_pseudoreport,0.711831,0.129500,0.000000,0.707193,0.172266,0.056497,14,non_transformer_gnn,gat,500,biomedclip_image_plus_medgemma_pseudoreport,0.5,0.5,/data/liangz2/openi/multi_kg/kg_non_transforme...,False,False,fixed_0.5,/data/liangz2/openi/multi_kg/medgemma_pseudore...
477,8,non_transformer_gnn_gat_medgemma_pseudoreport,0.711831,0.129500,0.000773,0.707193,0.172266,0.057214,14,non_transformer_gnn,gat,500,biomedclip_image_plus_medgemma_pseudoreport,0.5,0.5,/data/liangz2/openi/multi_kg/kg_non_transforme...,False,False,val_tuned_f1,/data/liangz2/openi/multi_kg/medgemma_pseudore...
478,9,non_transformer_gnn_gat_medgemma_pseudoreport,0.746906,0.146436,0.003097,0.710207,0.171488,0.059361,14,non_transformer_gnn,gat,500,biomedclip_image_plus_medgemma_pseudoreport,0.5,0.5,/data/liangz2/openi/multi_kg/kg_non_transforme...,False,False,fixed_0.5,/data/liangz2/openi/multi_kg/medgemma_pseudore...


## Aggregate Results

In [10]:
def collect_medgemma_gnn_results():
    files=[]; final=OUTPUT_ROOT/"medgemma_pseudoreport_gnn_all_results.csv"; progress=OUTPUT_ROOT/"medgemma_pseudoreport_gnn_all_results_progress.csv"
    if final.exists(): files.append(final)
    if progress.exists(): files.append(progress)
    files.extend(sorted(RESULT_ROOT.glob("*/*/top_k_*/fold*/metrics/*_medgemma_pseudoreport_method_summary.csv")))
    frames=[]
    for path in files:
        try: frame=pd.read_csv(path)
        except Exception as exc: print("Skipping",path,exc); continue
        if frame.empty or "fold" not in frame.columns or "threshold_mode" not in frame.columns: continue
        frame["source_file"]=str(path); frames.append(frame)
    if not frames: raise FileNotFoundError("No pseudo-report augmented GNN results found yet.")
    results=pd.concat(frames,ignore_index=True)
    keys=[c for c in ["family","model_name","query_top_k","fold","threshold_mode"] if c in results.columns]
    return results.sort_values("source_file").drop_duplicates(subset=keys,keep="last").reset_index(drop=True)

results=collect_medgemma_gnn_results(); collected_path=OUTPUT_ROOT/"medgemma_pseudoreport_gnn_all_results_collected.csv"; results.to_csv(collected_path,index=False)
print("Saved",collected_path,"rows",len(results))
metric_cols=["disease_macro_auroc","disease_macro_auprc","disease_macro_f1","all_macro_auroc","all_macro_auprc","all_macro_f1"]
rows=[]
for (family,model_name,top_k,mode),group in results.groupby(["family","model_name","query_top_k","threshold_mode"],sort=True):
    row={"family":family,"model_name":model_name,"query_top_k":int(top_k),"threshold_mode":mode,"n_folds":int(group["fold"].nunique()),"completed_folds":",".join(str(int(x)) for x in sorted(group["fold"].unique()))}
    for m in metric_cols: row[f"{m}_mean"]=group[m].mean(); row[f"{m}_std"]=group[m].std(ddof=1)
    rows.append(row)
agg=pd.DataFrame(rows); agg_path=OUTPUT_ROOT/"medgemma_pseudoreport_gnn_aggregate_summary.csv"; agg.to_csv(agg_path,index=False)
print("Saved",agg_path)
display(agg.sort_values(["threshold_mode","disease_macro_auroc_mean"],ascending=[True,False]))

Saved /data/liangz2/openi/multi_kg/medgemma_pseudoreport_gnn_inference/medgemma_pseudoreport_gnn_all_results_collected.csv rows 480
Saved /data/liangz2/openi/multi_kg/medgemma_pseudoreport_gnn_inference/medgemma_pseudoreport_gnn_aggregate_summary.csv


,family,model_name,query_top_k,threshold_mode,n_folds,completed_folds,disease_macro_auroc_mean,disease_macro_auroc_std,disease_macro_auprc_mean,disease_macro_auprc_std,disease_macro_f1_mean,disease_macro_f1_std,all_macro_auroc_mean,all_macro_auroc_std,all_macro_auprc_mean,all_macro_auprc_std,all_macro_f1_mean,all_macro_f1_std
20,non_transformer_gnn,graphsage,200,fixed_0.5,10,"0,1,2,3,4,5,6,7,8,9",0.770273,0.006969,0.148303,0.005508,0.108493,0.047070,0.761156,0.024117,0.192068,0.014872,0.147566,0.066393
42,transformer_gnn,transformerconv,100,fixed_0.5,10,"0,1,2,3,4,5,6,7,8,9",0.769796,0.006456,0.144133,0.005354,0.164215,0.033692,0.761000,0.016446,0.188336,0.010320,0.198349,0.041317
22,non_transformer_gnn,graphsage,500,fixed_0.5,10,"0,1,2,3,4,5,6,7,8,9",0.769689,0.005587,0.144746,0.004475,0.133641,0.061844,0.764937,0.011241,0.190219,0.008378,0.164253,0.071909
18,non_transformer_gnn,graphsage,100,fixed_0.5,10,"0,1,2,3,4,5,6,7,8,9",0.769156,0.005343,0.145635,0.004645,0.110732,0.044282,0.762436,0.017946,0.190721,0.012344,0.154751,0.049454
46,transformer_gnn,transformerconv,500,fixed_0.5,10,"0,1,2,3,4,5,6,7,8,9",0.768431,0.008878,0.144486,0.004695,0.149074,0.050234,0.760241,0.025553,0.188961,0.013571,0.184028,0.060739
16,non_transformer_gnn,graphsage,50,fixed_0.5,10,"0,1,2,3,4,5,6,7,8,9",0.767988,0.009945,0.146533,0.003174,0.145178,0.055905,0.750650,0.024580,0.184567,0.012790,0.185723,0.068254
40,transformer_gnn,transformerconv,50,fixed_0.5,10,"0,1,2,3,4,5,6,7,8,9",0.767346,0.009959,0.144699,0.004582,0.119259,0.076003,0.760291,0.018296,0.187972,0.008325,0.157519,0.086765
44,transformer_gnn,transformerconv,200,fixed_0.5,10,"0,1,2,3,4,5,6,7,8,9",0.766654,0.010854,0.143067,0.006523,0.125430,0.068673,0.758094,0.020297,0.186839,0.012006,0.156573,0.080931
36,transformer_gnn,hgtconv,200,fixed_0.5,10,"0,1,2,3,4,5,6,7,8,9",0.750837,0.022682,0.137469,0.010872,0.041167,0.056380,0.736644,0.033187,0.177126,0.017948,0.086695,0.058684
6,non_transformer_gnn,gat,500,fixed_0.5,10,"0,1,2,3,4,5,6,7,8,9",0.750364,0.031013,0.146034,0.010476,0.118582,0.070961,0.720776,0.015575,0.175630,0.007172,0.166601,0.065893


## Evaluate MedGemma Pseudo Reports

This safer evaluation cell computes lightweight report metrics first and saves them under `OUTPUT_ROOT / "pseudo_report_evaluation"`. `BERTScore` and `RadGraph-F1` are disabled by default because they load large NLP models and can crash the kernel on a full report set; enable them only after the lightweight metrics finish.


In [11]:
# Safe report-level evaluation for MedGemma pseudo reports.
# Run this cell after PSEUDO_REPORT_CSV has been created.
# Default behavior: compute lightweight metrics for all completed reports.
# Heavy model-based metrics are opt-in to avoid kernel crashes.

REPORT_EVAL_DIR = OUTPUT_ROOT / "pseudo_report_evaluation"
REPORT_EVAL_DIR.mkdir(parents=True, exist_ok=True)

PSEUDO_REPORT_EVAL_INPUT_CSV = REPORT_EVAL_DIR / "medgemma_1_5_4b_it_pseudo_report_eval_inputs.csv"
PSEUDO_REPORT_METRICS_CSV = REPORT_EVAL_DIR / "medgemma_1_5_4b_it_pseudo_report_metrics.csv"
PSEUDO_REPORT_SUMMARY_CSV = REPORT_EVAL_DIR / "medgemma_1_5_4b_it_pseudo_report_metric_summary.csv"

# Use None for all rows. Set to e.g. 200 for a quick smoke test.
MAX_REPORT_EVAL_ROWS = None

# Safe defaults. Turn these on one at a time after the basic metrics finish.
RUN_CIDER = True
RUN_BERTSCORE = False
RUN_RADGRAPH_F1 = False

# Safer optional BERTScore settings. Avoid roberta-large unless you have enough memory.
BERTSCORE_MODEL_TYPE = "distilroberta-base"
BERTSCORE_BATCH_SIZE = 8
BERTSCORE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BERTSCORE_MAX_ROWS = 512  # applied only when RUN_BERTSCORE=True; set None for all rows.

RADGRAPH_EVAL_MODEL = "modern-radgraph-xl"
RADGRAPH_EVAL_BATCH_SIZE = 4
RADGRAPH_MAX_ROWS = 512  # applied only when RUN_RADGRAPH_F1=True; set None for all rows.

REPORT_TEXT_KEYS = [
    "report_text", "reference_report", "findings", "impression", "report", "text",
    "Original_Caption", "Caption", "caption", "description", "summary",
]

NEGATION_TERMS = (
    "no", "not", "without", "absent", "negative for", "no evidence of",
    "no focal", "no acute", "free of",
)

LABEL_PATTERNS = {
    "normal": ["normal", "no acute cardiopulmonary abnormality", "no acute disease", "clear lungs"],
    "enlarged cardiomediastinum": ["enlarged cardiomediastinum", "mediastinal widening", "wide mediastinum"],
    "cardiomegaly": ["cardiomegaly", "enlarged heart", "cardiac enlargement", "heart is enlarged"],
    "lung opacity": ["lung opacity", "opacity", "opacities", "airspace opacity", "airspace disease"],
    "lung lesion": ["lung lesion", "nodule", "mass", "lesion"],
    "edema": ["edema", "pulmonary edema", "vascular congestion", "interstitial edema"],
    "consolidation": ["consolidation", "consolidative"],
    "pneumonia": ["pneumonia", "infectious process", "infection"],
    "atelectasis": ["atelectasis", "atelectatic", "volume loss"],
    "pneumothorax": ["pneumothorax", "pleural air"],
    "pleural effusion": ["pleural effusion", "effusion"],
    "pleural other": ["pleural thickening", "pleural plaque", "pleural abnormality", "pleural scarring"],
    "fracture": ["fracture", "rib fracture", "osseous fracture"],
    "support devices": ["support device", "support devices", "tube", "line", "catheter", "pacemaker", "endotracheal", "picc"],
}


def normalize_report_text(text):
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return ""
    text = str(text).replace("\r", " ").replace("\n", " ")
    return " ".join(text.split())


def safe_json_load(path):
    try:
        with open(path, "r") as f:
            return json.load(f)
    except Exception:
        return {}


def resolve_json_path_from_row(row):
    for col in ["json_path", "resolved_json_path"]:
        if col in row and pd.notna(row[col]) and str(row[col]).strip():
            path = Path(str(row[col]))
            if path.exists():
                return path
    for col in ["resolved_image_path", "image_path"]:
        if col in row and pd.notna(row[col]) and str(row[col]).strip():
            path = Path(str(row[col])).with_suffix(".json")
            if path.exists():
                return path
            alt = Path(str(path).replace("/vf/users/liangz2/openi", "/data/liangz2/openi"))
            if alt.exists():
                return alt
    return None


def collect_report_text_from_json(payload):
    parts = []
    for key in REPORT_TEXT_KEYS:
        value = payload.get(key)
        if isinstance(value, str) and value.strip():
            parts.append(value)
    for section_key in ["report", "report_sections", "sections"]:
        section = payload.get(section_key)
        if isinstance(section, dict):
            for key in ["indication", "history", "findings", "impression", "comparison"]:
                value = section.get(key) or section.get(key.upper()) or section.get(key.title())
                if isinstance(value, str) and value.strip():
                    parts.append(value)
    if not parts:
        bbox_ann = payload.get("bbox_annotation", {})
        if isinstance(bbox_ann, dict):
            for obs in bbox_ann.get("observation_bboxes", []) or []:
                sentence = obs.get("summary_sentence")
                if isinstance(sentence, str) and sentence.strip():
                    parts.append(sentence)
    return normalize_report_text(" ".join(parts))


def reference_report_from_row(row):
    for col in ["report_text", "reference_report"]:
        if col in row and pd.notna(row[col]) and str(row[col]).strip():
            return normalize_report_text(row[col])
    json_path = resolve_json_path_from_row(row)
    if json_path is None:
        return ""
    return collect_report_text_from_json(safe_json_load(json_path))


def parse_label_list(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, list):
        return [str(x).strip().lower() for x in value if str(x).strip()]
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return [str(x).strip().lower() for x in parsed if str(x).strip()]
    except Exception:
        pass
    return [x.strip().lower() for x in text.replace(";", ",").split(",") if x.strip()]


def reference_label_set(row):
    labels = set()
    for label, col in zip(SCORING_LABELS, TARGET_COLUMNS):
        if col in row and pd.notna(row[col]):
            try:
                if int(row[col]) == 1:
                    labels.add(label)
            except Exception:
                pass
    if not labels and "labels_normalized" in row:
        labels.update(x for x in parse_label_list(row["labels_normalized"]) if x in SCORING_LABELS)
    if not labels and str(row.get("normal", "")).strip().lower() == "yes":
        labels.add("normal")
    if labels - {"normal"}:
        labels.discard("normal")
    return labels


def phrase_is_negated(text, start_index):
    window = text[max(0, start_index - 55):start_index]
    return any(term in window for term in NEGATION_TERMS)


def generated_label_set(report_text):
    text = normalize_report_text(report_text).lower()
    labels = set()
    if not text:
        return labels
    for label, patterns in LABEL_PATTERNS.items():
        for pattern in patterns:
            index = text.find(pattern)
            if index >= 0 and not phrase_is_negated(text, index):
                labels.add(label)
                break
    if labels - {"normal"}:
        labels.discard("normal")
    return labels


def clinical_label_scores(reference_labels, generated_report):
    predicted_labels = generated_label_set(generated_report)
    y_true = np.array([1 if label in reference_labels else 0 for label in SCORING_LABELS], dtype=np.int32)
    y_pred = np.array([1 if label in predicted_labels else 0 for label in SCORING_LABELS], dtype=np.int32)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    pred_pos = int((y_pred == 1).sum())
    true_pos = int((y_true == 1).sum())
    precision = tp / pred_pos if pred_pos else 0.0
    recall = tp / true_pos if true_pos else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {
        "clinical_label_precision": precision,
        "clinical_label_recall": recall,
        "clinical_label_micro_f1": f1,
        "predicted_labels_from_report": json.dumps(sorted(predicted_labels)),
        "reference_labels": json.dumps(sorted(reference_labels)),
    }


def compute_bleu_scores(references, hypotheses):
    try:
        from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
    except Exception as exc:
        print(f"BLEU unavailable: {exc}")
        return [np.nan] * len(hypotheses)
    smoother = SmoothingFunction().method1
    scores = []
    for ref, hyp in zip(references, hypotheses):
        ref_tokens = normalize_report_text(ref).lower().split()
        hyp_tokens = normalize_report_text(hyp).lower().split()
        scores.append(float(sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smoother)) if ref_tokens and hyp_tokens else np.nan)
    return scores


def compute_rouge_l_scores(references, hypotheses):
    try:
        from rouge_score import rouge_scorer
    except Exception as exc:
        print(f"ROUGE-L unavailable: {exc}")
        return [np.nan] * len(hypotheses)
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    scores = []
    for ref, hyp in zip(references, hypotheses):
        scores.append(float(scorer.score(ref, hyp)["rougeL"].fmeasure) if normalize_report_text(ref) and normalize_report_text(hyp) else np.nan)
    return scores


def compute_cider_scores(references, hypotheses):
    if not RUN_CIDER:
        return [np.nan] * len(hypotheses)
    try:
        from pycocoevalcap.cider.cider import Cider
    except Exception as exc:
        print(f"CIDEr unavailable: {exc}")
        return [np.nan] * len(hypotheses)
    try:
        gts = {i: [normalize_report_text(ref)] for i, ref in enumerate(references)}
        res = {i: [normalize_report_text(hyp)] for i, hyp in enumerate(hypotheses)}
        _, scores = Cider().compute_score(gts, res)
        return [float(x) for x in scores]
    except Exception as exc:
        print(f"CIDEr failed: {exc}")
        return [np.nan] * len(hypotheses)


def compute_bertscore_f1(references, hypotheses):
    if not RUN_BERTSCORE:
        return [np.nan] * len(hypotheses)
    try:
        from bert_score import score as bert_score
    except Exception as exc:
        print(f"BERTScore unavailable: {exc}")
        return [np.nan] * len(hypotheses)
    rows = len(hypotheses) if BERTSCORE_MAX_ROWS is None else min(len(hypotheses), int(BERTSCORE_MAX_ROWS))
    scores = [np.nan] * len(hypotheses)
    try:
        for start in range(0, rows, BERTSCORE_BATCH_SIZE):
            end = min(start + BERTSCORE_BATCH_SIZE, rows)
            _, _, f1 = bert_score(
                hypotheses[start:end],
                references[start:end],
                lang="en",
                model_type=BERTSCORE_MODEL_TYPE,
                batch_size=BERTSCORE_BATCH_SIZE,
                device=BERTSCORE_DEVICE,
                verbose=False,
                rescale_with_baseline=False,
            )
            for offset, value in enumerate(f1.detach().cpu().numpy().tolist()):
                scores[start + offset] = float(value)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    except Exception as exc:
        print(f"BERTScore failed after partial scoring: {exc}")
    return scores


def radgraph_item(output, local_index):
    if isinstance(output, tuple):
        output = output[0]
    if isinstance(output, dict):
        key = str(local_index)
        if key in output:
            return output[key]
        values = list(output.values())
        return values[local_index] if local_index < len(values) else {}
    if isinstance(output, list):
        return output[local_index]
    return {}


def radgraph_entity_set(item):
    entities = item.get("entities", item) if isinstance(item, dict) else {}
    iterable = entities.values() if isinstance(entities, dict) else entities if isinstance(entities, list) else []
    result = set()
    for ent in iterable:
        if not isinstance(ent, dict):
            continue
        label = str(ent.get("label", ent.get("type", ""))).strip().lower()
        tokens = ent.get("tokens", ent.get("text", ent.get("span", "")))
        text = " ".join(str(x) for x in tokens) if isinstance(tokens, list) else str(tokens)
        text = normalize_report_text(text).lower()
        if text:
            result.add((text, label))
    return result


def compute_radgraph_f1(references, hypotheses):
    if not RUN_RADGRAPH_F1:
        return [np.nan] * len(hypotheses)
    try:
        from radgraph import RadGraph
        try:
            radgraph = RadGraph(model_type=RADGRAPH_EVAL_MODEL)
        except TypeError:
            radgraph = RadGraph(model=RADGRAPH_EVAL_MODEL)
    except Exception as exc:
        print(f"RadGraph-F1 unavailable: {exc}")
        return [np.nan] * len(hypotheses)
    rows = len(hypotheses) if RADGRAPH_MAX_ROWS is None else min(len(hypotheses), int(RADGRAPH_MAX_ROWS))
    scores = [np.nan] * len(hypotheses)
    for start in range(0, rows, RADGRAPH_EVAL_BATCH_SIZE):
        end = min(start + RADGRAPH_EVAL_BATCH_SIZE, rows)
        try:
            ref_output = radgraph(references[start:end])
            hyp_output = radgraph(hypotheses[start:end])
            for local_i in range(end - start):
                ref_entities = radgraph_entity_set(radgraph_item(ref_output, local_i))
                hyp_entities = radgraph_entity_set(radgraph_item(hyp_output, local_i))
                tp = len(ref_entities & hyp_entities)
                precision = tp / len(hyp_entities) if hyp_entities else 0.0
                recall = tp / len(ref_entities) if ref_entities else 0.0
                scores[start + local_i] = float(2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        except Exception as exc:
            print(f"RadGraph-F1 failed for rows {start}:{end}: {exc}")
    return scores


def build_report_eval_frame():
    reports = pseudo_reports_df.copy() if "pseudo_reports_df" in globals() else load_existing_pseudo_reports()
    if reports.empty:
        raise FileNotFoundError(f"No pseudo reports found. Expected {PSEUDO_REPORT_CSV}")
    reports = reports.copy()
    reports["row_id"] = reports["row_id"].astype(int)
    metadata = split_table_df.copy()
    metadata["row_id"] = metadata["row_id"].astype(int)
    merged = reports.merge(metadata, on="row_id", how="left", suffixes=("_pseudo", ""))
    if "status" in merged.columns:
        merged = merged[merged["status"].fillna("").astype(str).str.lower().eq("ok")].copy()
    merged["pseudo_report"] = merged["pseudo_report"].fillna("").map(normalize_report_text)
    merged = merged[merged["pseudo_report"].str.len() > 0].copy()
    if MAX_REPORT_EVAL_ROWS is not None:
        merged = merged.head(int(MAX_REPORT_EVAL_ROWS)).copy()
    merged["reference_report"] = [reference_report_from_row(row) for _, row in merged.iterrows()]
    merged = merged[merged["reference_report"].str.len() > 0].copy()
    if merged.empty:
        raise ValueError("No rows have both a pseudo report and a reference report.")
    merged.to_csv(PSEUDO_REPORT_EVAL_INPUT_CSV, index=False)
    return merged


def evaluate_medgemma_pseudo_reports():
    eval_frame = build_report_eval_frame()
    references = eval_frame["reference_report"].tolist()
    hypotheses = eval_frame["pseudo_report"].tolist()
    print(f"Evaluating {len(eval_frame)} pseudo reports")
    print(f"Heavy metrics: BERTScore={RUN_BERTSCORE}, RadGraph-F1={RUN_RADGRAPH_F1}")

    metrics = pd.DataFrame({
        "row_id": eval_frame["row_id"].astype(int).values,
        "study_id": eval_frame.get("study_id", pd.Series([None] * len(eval_frame))).values,
        "image_path": eval_frame.get("image_path", pd.Series([None] * len(eval_frame))).values,
        "bleu": compute_bleu_scores(references, hypotheses),
        "rouge_l": compute_rouge_l_scores(references, hypotheses),
        "cider": compute_cider_scores(references, hypotheses),
        "bertscore_f1": compute_bertscore_f1(references, hypotheses),
        "radgraph_f1": compute_radgraph_f1(references, hypotheses),
    })

    clinical_df = pd.DataFrame([
        clinical_label_scores(reference_label_set(row), row["pseudo_report"])
        for _, row in eval_frame.iterrows()
    ])
    metrics = pd.concat([metrics.reset_index(drop=True), clinical_df.reset_index(drop=True)], axis=1)
    metrics.to_csv(PSEUDO_REPORT_METRICS_CSV, index=False)

    numeric_cols = [
        "bleu", "rouge_l", "cider", "bertscore_f1", "radgraph_f1",
        "clinical_label_micro_f1", "clinical_label_precision", "clinical_label_recall",
    ]
    summary = metrics[numeric_cols].agg(["count", "mean", "std", "median", "min", "max"]).T.reset_index()
    summary = summary.rename(columns={"index": "metric"})
    summary.to_csv(PSEUDO_REPORT_SUMMARY_CSV, index=False)

    print("Saved evaluation inputs:", PSEUDO_REPORT_EVAL_INPUT_CSV)
    print("Saved per-report metrics:", PSEUDO_REPORT_METRICS_CSV)
    print("Saved metric summary:", PSEUDO_REPORT_SUMMARY_CSV)
    display(summary)
    return metrics, summary


pseudo_report_metrics_df, pseudo_report_summary_df = evaluate_medgemma_pseudo_reports()


Evaluating 44100 pseudo reports
Heavy metrics: BERTScore=False, RadGraph-F1=False
Saved evaluation inputs: /data/liangz2/openi/multi_kg/medgemma_pseudoreport_gnn_inference/pseudo_report_evaluation/medgemma_1_5_4b_it_pseudo_report_eval_inputs.csv
Saved per-report metrics: /data/liangz2/openi/multi_kg/medgemma_pseudoreport_gnn_inference/pseudo_report_evaluation/medgemma_1_5_4b_it_pseudo_report_metrics.csv
Saved metric summary: /data/liangz2/openi/multi_kg/medgemma_pseudoreport_gnn_inference/pseudo_report_evaluation/medgemma_1_5_4b_it_pseudo_report_metric_summary.csv


,metric,count,mean,std,median,min,max
0,bleu,44100.0,0.008751,0.010820,4.791580e-03,2.890777e-14,0.161235
1,rouge_l,44100.0,0.173919,0.048573,1.714286e-01,2.439024e-02,0.412214
2,cider,44100.0,0.000292,0.008625,2.012506e-81,0.000000e+00,0.731789
3,bertscore_f1,0.0,NaN,NaN,NaN,NaN,NaN
4,radgraph_f1,0.0,NaN,NaN,NaN,NaN,NaN
5,clinical_label_micro_f1,44100.0,0.527196,0.450957,5.000000e-01,0.000000e+00,1.000000
6,clinical_label_precision,44100.0,0.509680,0.455973,5.000000e-01,0.000000e+00,1.000000
7,clinical_label_recall,44100.0,0.582229,0.468301,1.000000e+00,0.000000e+00,1.000000


## Generate MedGemma Reports with GNN Graph-RAG Evidence

This cell loads only the best non-transformer GNN and the best transformer GNN, runs fold-specific inference, converts each prediction plus neighbor evidence into a structured graph prompt, and asks MedGemma to generate a graph-augmented report. The outputs are resumable CSV files, so interrupted runs continue from completed `(config, fold, row_id)` records.


In [12]:
# Generate GNN-RAG reports with MedGemma.
# The prompt includes the image plus structured graph evidence from the trained GNN and retrieved KG neighbors.

GNN_RAG_REPORT_ROOT = OUTPUT_ROOT / "gnn_rag_reports"
GNN_RAG_REPORT_ROOT.mkdir(parents=True, exist_ok=True)

BEST_GNN_SUMMARY_CANDIDATES = [
    DATA_ROOT / "gnn_vs_biomedclip_comparison" / "best_method_summary.csv",
    Path.cwd() / "gnn_vs_biomedclip_comparison" / "best_method_summary.csv",
    Path("/Users/liangz2/Documents/inverse_diffusion/gnn_vs_biomedclip_comparison/best_method_summary.csv"),
]

GNN_AGGREGATE_SUMMARY_CANDIDATES = {
    "non_transformer_gnn": [
        DATA_ROOT / "kg_non_transformer_gnn_10fold" / "non_transformer_gnn_10fold_aggregate_summary.csv",
        DATA_ROOT / "non_transformer_gnn_10fold" / "non_transformer_gnn_10fold_aggregate_summary.csv",
        Path.cwd() / "non_transformer_gnn_10fold" / "non_transformer_gnn_10fold_aggregate_summary.csv",
        Path("/Users/liangz2/Documents/inverse_diffusion/non_transformer_gnn_10fold/non_transformer_gnn_10fold_aggregate_summary.csv"),
    ],
    "transformer_gnn": [
        DATA_ROOT / "kg_transformer_gnn_10fold" / "transformer_gnn_10fold_aggregate_summary.csv",
        DATA_ROOT / "transformer_gnn_10fold" / "transformer_gnn_10fold_aggregate_summary.csv",
        Path.cwd() / "transformer_gnn_10fold" / "transformer_gnn_10fold_aggregate_summary.csv",
        Path("/Users/liangz2/Documents/inverse_diffusion/transformer_gnn_10fold/transformer_gnn_10fold_aggregate_summary.csv"),
    ],
}

GNN_RAG_FALLBACK_CONFIGS = [
    {"group": "Best non-transformer GNN", "family": "non_transformer_gnn", "model_name": "graphsage", "query_top_k": 500},
    {"group": "Best transformer GNN", "family": "transformer_gnn", "model_name": "transformerconv", "query_top_k": 100},
]

GNN_RAG_FOLD_INDICES = FOLD_INDICES
GNN_RAG_MAX_REPORTS_PER_CONFIG = None  # set e.g. 200 for a smoke test
GNN_RAG_SAVE_EVERY = 10
GNN_RAG_TOP_FINDINGS_IN_PROMPT = 7
GNN_RAG_TOP_NEIGHBOR_LABELS_IN_PROMPT = 7
GNN_RAG_NUM_NEIGHBOR_EXAMPLES_IN_PROMPT = 3
GNN_RAG_TOP_KG_ENTITIES_IN_PROMPT = 10
GNN_RAG_NEIGHBOR_TEMPERATURE = 0.05
GNN_RAG_REUSE_EXISTING = True


def first_existing_path(paths):
    return next((p for p in paths if Path(p).exists()), None)


def config_name_from_parts(family, model_name, top_k):
    return f"{family}_{model_name}_topk{int(top_k)}"


def normalize_gnn_config(row):
    family = str(row["family"])
    model_name = str(row["model_name"])
    top_k = int(float(row["query_top_k"]))
    group = str(row.get("group", f"{family} {model_name}"))
    return {
        "group": group,
        "family": family,
        "model_name": model_name,
        "query_top_k": top_k,
        "config_name": config_name_from_parts(family, model_name, top_k),
    }


def best_config_from_aggregate(family):
    path = first_existing_path(GNN_AGGREGATE_SUMMARY_CANDIDATES.get(family, []))
    if path is None:
        return None
    df = pd.read_csv(path)
    if "threshold_mode" in df.columns:
        fixed = df[df["threshold_mode"].astype(str).eq("fixed_0.5")].copy()
        if len(fixed):
            df = fixed
    df = df.dropna(subset=["disease_macro_auroc_mean", "query_top_k"])
    if df.empty:
        return None
    best = df.sort_values("disease_macro_auroc_mean", ascending=False).iloc[0].to_dict()
    best["family"] = family
    best["group"] = "Best non-transformer GNN" if family == "non_transformer_gnn" else "Best transformer GNN"
    return normalize_gnn_config(best)


def select_best_gnn_rag_configs():
    selected = []
    best_path = first_existing_path(BEST_GNN_SUMMARY_CANDIDATES)
    if best_path is not None:
        df = pd.read_csv(best_path)
        wanted_groups = {"Best non-transformer GNN", "Best transformer GNN"}
        part = df[df["group"].astype(str).isin(wanted_groups)].copy() if "group" in df.columns else pd.DataFrame()
        for _, row in part.iterrows():
            selected.append(normalize_gnn_config(row.to_dict()))
        print("Loaded best GNN config summary:", best_path)

    families_found = {cfg["family"] for cfg in selected}
    for family in ["non_transformer_gnn", "transformer_gnn"]:
        if family not in families_found:
            cfg = best_config_from_aggregate(family)
            if cfg is not None:
                selected.append(cfg)

    families_found = {cfg["family"] for cfg in selected}
    for fallback in GNN_RAG_FALLBACK_CONFIGS:
        if fallback["family"] not in families_found:
            selected.append(normalize_gnn_config(fallback))

    # Keep one config per family, preserving priority order.
    out = []
    seen = set()
    for cfg in selected:
        if cfg["family"] not in seen:
            out.append(cfg)
            seen.add(cfg["family"])
    return out


BEST_GNN_RAG_CONFIGS = select_best_gnn_rag_configs()
print(pd.DataFrame(BEST_GNN_RAG_CONFIGS))


def gnn_rag_report_paths(config):
    root = GNN_RAG_REPORT_ROOT / config["config_name"]
    root.mkdir(parents=True, exist_ok=True)
    return {
        "root": root,
        "progress": root / f"{config['config_name']}_medgemma_gnn_rag_reports_progress.csv",
        "final": root / f"{config['config_name']}_medgemma_gnn_rag_reports.csv",
    }


def empty_gnn_rag_report_frame():
    return pd.DataFrame(columns=[
        "config_name", "family", "model_name", "query_top_k", "fold", "row_id", "study_id",
        "image_name", "image_path", "resolved_image_path", "gnn_rag_report", "gnn_prompt",
        "gnn_top_findings", "neighbor_label_evidence", "status", "error",
    ])


def load_existing_gnn_rag_reports(config):
    paths = gnn_rag_report_paths(config)
    frames = []
    for path in [paths["progress"], paths["final"]]:
        if path.exists():
            try:
                frames.append(pd.read_csv(path))
                print("Loaded GNN-RAG report checkpoint:", path)
            except Exception as exc:
                print("Skipping unreadable GNN-RAG checkpoint:", path, repr(exc))
    if not frames:
        return empty_gnn_rag_report_frame()
    out = pd.concat(frames, ignore_index=True, sort=False)
    for col in ["fold", "row_id"]:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    out = out.dropna(subset=["fold", "row_id"]).copy()
    out["fold"] = out["fold"].astype(int)
    out["row_id"] = out["row_id"].astype(int)
    return out.drop_duplicates(["config_name", "fold", "row_id"], keep="last").reset_index(drop=True)


def labels_from_target_vector(vector, include_normal=True):
    labels = []
    for label, value in zip(SCORING_LABELS, vector):
        if float(value) >= 0.5 and (include_normal or label != "normal"):
            labels.append(label)
    if any(label != "normal" for label in labels):
        labels = [label for label in labels if label != "normal"]
    return labels


def softmax_weights_from_sims(sims, temperature=GNN_RAG_NEIGHBOR_TEMPERATURE):
    sims = np.asarray(sims, dtype=np.float32)
    if sims.size == 0:
        return sims
    temp = max(float(temperature), 1e-6)
    scaled = (sims - np.max(sims)) / temp
    weights = np.exp(np.clip(scaled, -50, 50))
    denom = float(weights.sum())
    return weights / denom if denom > 0 else np.full_like(weights, 1.0 / len(weights))


def neighbor_label_prior(train_targets, neighbor_indices, neighbor_sims):
    weights = softmax_weights_from_sims(neighbor_sims)
    if len(weights) == 0:
        return np.zeros((len(SCORING_LABELS),), dtype=np.float32)
    return (train_targets[neighbor_indices].astype(np.float32) * weights[:, None]).sum(axis=0).astype(np.float32)


def format_gnn_top_findings(probs, thresholds):
    rows = []
    for label, prob, thr in zip(SCORING_LABELS, probs, thresholds):
        rows.append({"label": label, "probability": float(prob), "threshold": float(thr), "positive": bool(prob >= thr)})
    rows = sorted(rows, key=lambda x: x["probability"], reverse=True)
    positive = [r for r in rows if r["positive"] and r["label"] != "normal"]
    top = positive[:GNN_RAG_TOP_FINDINGS_IN_PROMPT] if positive else [r for r in rows if r["label"] != "normal"][:GNN_RAG_TOP_FINDINGS_IN_PROMPT]
    return top


def format_neighbor_label_evidence(prior):
    pairs = [
        {"label": label, "support": float(score)}
        for label, score in zip(SCORING_LABELS, prior)
        if label != "normal"
    ]
    return sorted(pairs, key=lambda x: x["support"], reverse=True)[:GNN_RAG_TOP_NEIGHBOR_LABELS_IN_PROMPT]


def format_neighbor_examples(train_df, train_targets, neighbor_indices, neighbor_sims):
    rows = []
    for rank, idx_value in enumerate(neighbor_indices[:GNN_RAG_NUM_NEIGHBOR_EXAMPLES_IN_PROMPT], start=1):
        row = train_df.iloc[int(idx_value)]
        labels = labels_from_target_vector(train_targets[int(idx_value)], include_normal=False)
        rows.append({
            "rank": rank,
            "similarity": float(neighbor_sims[rank - 1]),
            "study_id": str(row.get("study_id", "")),
            "labels": labels,
        })
    return rows


def node_display_names(data, node_type):
    store = data[node_type]
    for attr in ["canonical_name", "label_name", "node_id"]:
        if hasattr(store, attr):
            values = list(getattr(store, attr))
            return [str(x).replace("observation::", "").replace("entity_term::", "").replace("anatomy::", "").replace("label::", "") for x in values]
    return [f"{node_type}_{i}" for i in range(int(store.x.size(0)))]


def collect_neighbor_kg_entity_evidence(data, neighbor_indices, neighbor_sims):
    """Collect graph entity/anatomy/observation terms attached to retrieved neighbor images.

    This keeps the prompt graph-grounded when the KG contains report/entity edges. If the
    loaded KG does not expose those node/edge types, the function safely returns an empty list.
    """
    if not hasattr(data, "edge_types"):
        return []
    weights = softmax_weights_from_sims(neighbor_sims)
    neighbor_weight = {int(idx_value): float(weights[pos]) for pos, idx_value in enumerate(neighbor_indices)}
    candidate_dst_types = {"observation", "entity_term", "anatomy", "finding", "attribute", "assertion"}
    blocked_relations = {"similar_to", "rev_similar_to", "has_finding", "rev_has_finding"}
    scores = {}
    for edge_type in data.edge_types:
        src_type, relation, dst_type = edge_type
        if src_type != "image" or dst_type not in candidate_dst_types or relation in blocked_relations:
            continue
        if not hasattr(data[edge_type], "edge_index"):
            continue
        edge_index = data[edge_type].edge_index.cpu().numpy()
        names = node_display_names(data, dst_type)
        for src_i, dst_i in zip(edge_index[0], edge_index[1]):
            src_i = int(src_i)
            if src_i not in neighbor_weight:
                continue
            dst_i = int(dst_i)
            if dst_i < 0 or dst_i >= len(names):
                continue
            key = (dst_type, names[dst_i])
            scores[key] = scores.get(key, 0.0) + neighbor_weight[src_i]
    rows = [
        {"node_type": node_type, "term": term, "support": float(score)}
        for (node_type, term), score in scores.items()
        if term and term.lower() not in {"nan", "none"}
    ]
    rows = sorted(rows, key=lambda x: x["support"], reverse=True)
    return rows[:GNN_RAG_TOP_KG_ENTITIES_IN_PROMPT]


def build_gnn_rag_prompt(top_findings, neighbor_evidence, neighbor_examples, kg_entity_evidence=None):
    finding_lines = [
        f"- {item['label']}: probability {item['probability']:.3f}, threshold {item['threshold']:.3f}, {'above' if item['positive'] else 'below'} threshold"
        for item in top_findings
    ]
    neighbor_lines = [f"- {item['label']}: weighted neighbor support {item['support']:.3f}" for item in neighbor_evidence]
    example_lines = []
    for item in neighbor_examples:
        labels = ", ".join(item["labels"]) if item["labels"] else "normal/no listed positive CheXpert finding"
        example_lines.append(f"- neighbor {item['rank']}: similarity {item['similarity']:.3f}; labels: {labels}")
    entity_lines = []
    for item in kg_entity_evidence or []:
        entity_lines.append(f"- {item['node_type']}: {item['term']} (support {item['support']:.3f})")

    return f"""Generate a concise radiology-style report for this frontal chest radiograph.
Use exactly two sections: FINDINGS and IMPRESSION.

You are also given graph-based retrieval evidence from a chest X-ray knowledge graph. Use it as supporting context, but prioritize the visible image. Do not mention probabilities, retrieval, graph models, or this prompt in the final report.

GNN predicted finding evidence:
{chr(10).join(finding_lines) if finding_lines else '- no strong graph finding'}

Retrieved-neighbor label support:
{chr(10).join(neighbor_lines) if neighbor_lines else '- no neighbor label support'}

Knowledge-graph entity support from retrieved studies:
{chr(10).join(entity_lines) if entity_lines else '- no extra graph entity evidence available'}

Nearest training-study examples:
{chr(10).join(example_lines) if example_lines else '- no neighbor examples'}

Mention cardiomegaly, edema, lung opacity, consolidation, pneumonia, atelectasis, pneumothorax, pleural effusion, fracture, and support devices only when supported by the image and evidence. If the image appears normal, say no acute cardiopulmonary abnormality.
"""


def load_prediction_probs_if_available(config, fold, test_df):
    dirs = output_dirs(config["family"], config["model_name"], config["query_top_k"], fold)
    pred_path = dirs["predictions"] / f"{config['model_name']}_medgemma_pseudoreport_test_predictions.csv"
    if not pred_path.exists():
        return None
    pred_df = pd.read_csv(pred_path)
    if len(pred_df) != len(test_df):
        return None
    prob_cols = [f"prob_{label}" for label in SCORING_LABELS]
    if not all(col in pred_df.columns for col in prob_cols):
        return None
    print("Using cached GNN probabilities:", pred_path)
    return pred_df[prob_cols].to_numpy(dtype=np.float32)


def infer_gnn_probs_for_fold(config, fold, data, train_df, test_df, test_image_embeddings, sims, idx):
    cached = load_prediction_probs_if_available(config, fold, test_df)
    if cached is not None:
        return cached

    ckpt_path = checkpoint_path(config["family"], config["model_name"], config["query_top_k"], fold)
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Missing GNN checkpoint: {ckpt_path}")
    ckpt = torch_load_any(ckpt_path, map_location="cpu")
    full_edges = message_passing_edge_index_dict(data)
    metadata = (data.node_types, list(full_edges.keys()))
    input_dims = input_dims_from_data(data)
    model = build_model(config["model_name"], metadata, input_dims, ckpt).to(DEVICE)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()

    test_targets = target_matrix_from_metadata(test_df)
    chunks = []
    for s in tqdm(range(0, len(test_df), QUERY_INFERENCE_CHUNK_SIZE), desc=f"GNN probs {config['config_name']} fold={fold}"):
        e = min(s + QUERY_INFERENCE_CHUNK_SIZE, len(test_df))
        sub, q_local, label_indices = build_query_subgraph(data, test_image_embeddings[s:e], test_targets[s:e], idx[s:e], sims[s:e])
        sub_edges = message_passing_edge_index_dict(sub)
        sub = sub.to(DEVICE)
        sub_edges = {et: ei.to(DEVICE) for et, ei in sub_edges.items()}
        with torch.no_grad():
            z = model(sub.x_dict, sub_edges)
            logits = model.decode_image_label_logits(z, label_indices)
            chunk = logits[torch.tensor(q_local, device=logits.device)].detach().cpu().numpy().astype("float32")
            chunks.append(sigmoid_np(chunk))
        del sub, sub_edges, z, logits
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return np.concatenate(chunks, axis=0)


def prepare_gnn_rag_fold_context(config, fold):
    data, train_df, test_df, test_image_embeddings = load_fold_data(fold)
    train_ids = frame_row_ids(train_df)
    test_ids = frame_row_ids(test_df)
    train_report_embeddings = pseudo_report_text_embeddings[train_ids]
    test_report_embeddings = pseudo_report_text_embeddings[test_ids]
    train_image_embeddings = data["image"].x.cpu().numpy().astype("float32")
    train_ret = combine_image_report_embeddings(train_image_embeddings, train_report_embeddings)
    test_ret = combine_image_report_embeddings(test_image_embeddings, test_report_embeddings)
    sims, idx = topk_inner_product(train_ret, test_ret, top_k=config["query_top_k"])
    probs = infer_gnn_probs_for_fold(config, fold, data, train_df, test_df, test_image_embeddings, sims, idx)
    thresholds = load_thresholds(threshold_path(config["family"], config["model_name"], config["query_top_k"], fold))
    return data, train_df.reset_index(drop=True), test_df.reset_index(drop=True), probs, sims, idx, thresholds


def generate_gnn_rag_reports_for_config(config):
    paths = gnn_rag_report_paths(config)
    existing = load_existing_gnn_rag_reports(config) if GNN_RAG_REUSE_EXISTING else empty_gnn_rag_report_frame()
    done = set()
    if len(existing) and "status" in existing.columns:
        ok = existing[existing["status"].fillna("").astype(str).str.lower().eq("ok")]
        done = set(zip(ok["fold"].astype(int), ok["row_id"].astype(int)))
    rows = existing.to_dict(orient="records") if len(existing) else []
    generated_total = 0

    for fold in GNN_RAG_FOLD_INDICES:
        ckpt_path = checkpoint_path(config["family"], config["model_name"], config["query_top_k"], fold)
        if not ckpt_path.exists():
            print("Missing checkpoint, skipping fold:", ckpt_path)
            continue
        print(f"\n===== GNN-RAG generation {config['config_name']} fold={fold} =====")
        data, train_df, test_df, probs, sims, idx, thresholds = prepare_gnn_rag_fold_context(config, fold)
        train_targets = target_matrix_from_metadata(train_df)
        test_ids = frame_row_ids(test_df)

        for local_i, row in tqdm(list(test_df.iterrows()), total=len(test_df), desc=f"MedGemma GNN-RAG {config['config_name']} fold={fold}"):
            row_id = int(test_ids[int(local_i)])
            if (fold, row_id) in done:
                continue
            if GNN_RAG_MAX_REPORTS_PER_CONFIG is not None and generated_total >= int(GNN_RAG_MAX_REPORTS_PER_CONFIG):
                break
            top_findings = format_gnn_top_findings(probs[int(local_i)], thresholds)
            prior = neighbor_label_prior(train_targets, idx[int(local_i)], sims[int(local_i)])
            neighbor_evidence = format_neighbor_label_evidence(prior)
            neighbor_examples = format_neighbor_examples(train_df, train_targets, idx[int(local_i)], sims[int(local_i)])
            kg_entity_evidence = collect_neighbor_kg_entity_evidence(data, idx[int(local_i)], sims[int(local_i)])
            prompt = build_gnn_rag_prompt(top_findings, neighbor_evidence, neighbor_examples, kg_entity_evidence=kg_entity_evidence)
            out = {
                "config_name": config["config_name"],
                "family": config["family"],
                "model_name": config["model_name"],
                "query_top_k": int(config["query_top_k"]),
                "fold": int(fold),
                "row_id": row_id,
                "study_id": str(row.get("study_id", "")),
                "image_name": str(row.get("image_name", Path(str(row.get("image_path", ""))).name)),
                "image_path": str(row.get("image_path", "")),
                "resolved_image_path": resolve_image_path(str(row.get("image_path", "")), row),
                "gnn_prompt": prompt,
                "gnn_top_findings": json.dumps(top_findings),
                "neighbor_label_evidence": json.dumps(neighbor_evidence),
                "kg_entity_evidence": json.dumps(kg_entity_evidence),
            }
            try:
                out["gnn_rag_report"] = generate_medgemma_report(out["resolved_image_path"], prompt=prompt)
                out["status"] = "ok"
                out["error"] = ""
            except Exception as exc:
                out["gnn_rag_report"] = ""
                out["status"] = "error"
                out["error"] = repr(exc)
            rows.append(out)
            generated_total += 1
            if generated_total % GNN_RAG_SAVE_EVERY == 0:
                pd.DataFrame(rows).drop_duplicates(["config_name", "fold", "row_id"], keep="last").to_csv(paths["progress"], index=False)

        pd.DataFrame(rows).drop_duplicates(["config_name", "fold", "row_id"], keep="last").to_csv(paths["progress"], index=False)
        del data
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if GNN_RAG_MAX_REPORTS_PER_CONFIG is not None and generated_total >= int(GNN_RAG_MAX_REPORTS_PER_CONFIG):
            print("Reached GNN_RAG_MAX_REPORTS_PER_CONFIG", GNN_RAG_MAX_REPORTS_PER_CONFIG)
            break

    final_df = pd.DataFrame(rows).drop_duplicates(["config_name", "fold", "row_id"], keep="last") if rows else empty_gnn_rag_report_frame()
    final_df.to_csv(paths["final"], index=False)
    print("Saved GNN-RAG reports:", paths["final"], final_df.shape)
    return final_df


all_gnn_rag_report_frames = []
for cfg in BEST_GNN_RAG_CONFIGS:
    all_gnn_rag_report_frames.append(generate_gnn_rag_reports_for_config(cfg))

gnn_rag_reports_df = pd.concat(all_gnn_rag_report_frames, ignore_index=True, sort=False) if all_gnn_rag_report_frames else empty_gnn_rag_report_frame()
gnn_rag_all_path = GNN_RAG_REPORT_ROOT / "best_gnn_rag_reports_all_configs.csv"
gnn_rag_reports_df.to_csv(gnn_rag_all_path, index=False)
print("Saved combined GNN-RAG reports:", gnn_rag_all_path, gnn_rag_reports_df.shape)



                      group               family       model_name  \
0  Best non-transformer GNN  non_transformer_gnn        graphsage   
1      Best transformer GNN      transformer_gnn  transformerconv   

   query_top_k                              config_name  
0          500    non_transformer_gnn_graphsage_topk500  
1          100  transformer_gnn_transformerconv_topk100  
Loaded GNN-RAG report checkpoint: /data/liangz2/openi/multi_kg/medgemma_pseudoreport_gnn_inference/gnn_rag_reports/non_transformer_gnn_graphsage_topk500/non_transformer_gnn_graphsage_topk500_medgemma_gnn_rag_reports_progress.csv

===== GNN-RAG generation non_transformer_gnn_graphsage_topk500 fold=0 =====


pseudo-report FAISS top_k=500:   0%|          | 0/9 [00:00<?, ?it/s]

Using cached GNN probabilities: /data/liangz2/openi/multi_kg/medgemma_pseudoreport_gnn_inference/gnn_inference_results/non_transformer_gnn/graphsage/top_k_500/fold0/predictions/graphsage_medgemma_pseudoreport_test_predictions.csv


MedGemma GNN-RAG non_transformer_gnn_graphsage_topk500 fold=0:   0%|          | 0/4410 [00:00<?, ?it/s]


===== GNN-RAG generation non_transformer_gnn_graphsage_topk500 fold=1 =====


...

pseudo-report FAISS top_k=100:   0%|          | 0/9 [00:00<?, ?it/s]

Using cached GNN probabilities: /data/liangz2/openi/multi_kg/medgemma_pseudoreport_gnn_inference/gnn_inference_results/transformer_gnn/transformerconv/top_k_100/fold8/predictions/transformerconv_medgemma_pseudoreport_test_predictions.csv


MedGemma GNN-RAG transformer_gnn_transformerconv_topk100 fold=8:   0%|          | 0/4410 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[tra

KeyboardInterrupt: 

## Evaluate GNN-RAG Reports and Compare with Direct MedGemma

This cell evaluates the graph-augmented MedGemma reports using the same report metrics as the direct MedGemma pseudo reports, then writes paired comparison tables. Run the safe direct-report evaluation cell first so the shared metric helpers are available.


In [ ]:
# Evaluate GNN-RAG reports and compare them with direct MedGemma pseudo reports.
# This cell reuses the metric helper functions from the direct MedGemma report-evaluation cell.

GNN_RAG_EVAL_ROOT = OUTPUT_ROOT / "gnn_rag_report_evaluation"
GNN_RAG_EVAL_ROOT.mkdir(parents=True, exist_ok=True)

GNN_RAG_REPORT_EVAL_MAX_ROWS = None  # set e.g. 500 for a quick smoke test

required_report_eval_helpers = [
    "normalize_report_text", "reference_report_from_row", "reference_label_set", "clinical_label_scores",
    "compute_bleu_scores", "compute_rouge_l_scores", "compute_cider_scores", "compute_bertscore_f1", "compute_radgraph_f1",
]
missing_helpers = [name for name in required_report_eval_helpers if name not in globals()]
if missing_helpers:
    raise RuntimeError(
        "Run the 'Evaluate MedGemma Pseudo Reports' cell first so these helpers are defined: "
        + ", ".join(missing_helpers)
    )


def collect_gnn_rag_report_files():
    if "BEST_GNN_RAG_CONFIGS" in globals():
        configs = BEST_GNN_RAG_CONFIGS
    else:
        configs = select_best_gnn_rag_configs()
    files = []
    for cfg in configs:
        paths = gnn_rag_report_paths(cfg)
        if paths["final"].exists():
            files.append((cfg, paths["final"]))
        elif paths["progress"].exists():
            files.append((cfg, paths["progress"]))
        else:
            print("No GNN-RAG report file found for", cfg["config_name"])
    return files


def build_gnn_rag_eval_frame(config, report_path):
    reports = pd.read_csv(report_path)
    if reports.empty:
        return reports
    reports = reports.copy()
    reports["row_id"] = reports["row_id"].astype(int)
    reports["fold"] = reports["fold"].astype(int)
    if "status" in reports.columns:
        reports = reports[reports["status"].fillna("").astype(str).str.lower().eq("ok")].copy()
    reports["gnn_rag_report"] = reports["gnn_rag_report"].fillna("").map(normalize_report_text)
    reports = reports[reports["gnn_rag_report"].str.len() > 0].copy()
    if GNN_RAG_REPORT_EVAL_MAX_ROWS is not None:
        reports = reports.head(int(GNN_RAG_REPORT_EVAL_MAX_ROWS)).copy()

    metadata = split_table_df.copy()
    metadata["row_id"] = metadata["row_id"].astype(int)
    merged = reports.merge(metadata, on="row_id", how="left", suffixes=("_rag", ""))
    merged["reference_report"] = [reference_report_from_row(row) for _, row in merged.iterrows()]
    merged = merged[merged["reference_report"].str.len() > 0].copy()
    merged["config_name"] = config["config_name"]
    merged["family"] = config["family"]
    merged["model_name"] = config["model_name"]
    merged["query_top_k"] = int(config["query_top_k"])
    return merged


def evaluate_report_dataframe(eval_frame, report_column, method_name, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    references = eval_frame["reference_report"].tolist()
    hypotheses = eval_frame[report_column].tolist()
    print(f"Evaluating {method_name}: {len(eval_frame)} reports")

    metrics = pd.DataFrame({
        "method": method_name,
        "row_id": eval_frame["row_id"].astype(int).values,
        "fold": eval_frame.get("fold", pd.Series([np.nan] * len(eval_frame))).values,
        "study_id": eval_frame.get("study_id", pd.Series([None] * len(eval_frame))).values,
        "image_path": eval_frame.get("image_path", pd.Series([None] * len(eval_frame))).values,
        "bleu": compute_bleu_scores(references, hypotheses),
        "rouge_l": compute_rouge_l_scores(references, hypotheses),
        "cider": compute_cider_scores(references, hypotheses),
        "bertscore_f1": compute_bertscore_f1(references, hypotheses),
        "radgraph_f1": compute_radgraph_f1(references, hypotheses),
    })
    clinical_df = pd.DataFrame([
        clinical_label_scores(reference_label_set(row), row[report_column])
        for _, row in eval_frame.iterrows()
    ])
    metrics = pd.concat([metrics.reset_index(drop=True), clinical_df.reset_index(drop=True)], axis=1)

    numeric_cols = [
        "bleu", "rouge_l", "cider", "bertscore_f1", "radgraph_f1",
        "clinical_label_micro_f1", "clinical_label_precision", "clinical_label_recall",
    ]
    summary = metrics[numeric_cols].agg(["count", "mean", "std", "median", "min", "max"]).T.reset_index()
    summary = summary.rename(columns={"index": "metric"})
    summary.insert(0, "method", method_name)

    metrics_path = output_dir / f"{method_name}_report_metrics.csv"
    summary_path = output_dir / f"{method_name}_report_metric_summary.csv"
    metrics.to_csv(metrics_path, index=False)
    summary.to_csv(summary_path, index=False)
    print("Saved", metrics_path)
    print("Saved", summary_path)
    return metrics, summary


def load_or_evaluate_direct_medgemma_metrics():
    if PSEUDO_REPORT_METRICS_CSV.exists() and PSEUDO_REPORT_SUMMARY_CSV.exists():
        direct_metrics = pd.read_csv(PSEUDO_REPORT_METRICS_CSV)
        direct_summary = pd.read_csv(PSEUDO_REPORT_SUMMARY_CSV)
        direct_metrics.insert(0, "method", "direct_medgemma") if "method" not in direct_metrics.columns else None
        direct_summary.insert(0, "method", "direct_medgemma") if "method" not in direct_summary.columns else None
        return direct_metrics, direct_summary
    print("Direct MedGemma report metrics not found; evaluating direct pseudo reports now.")
    direct_metrics, direct_summary = evaluate_medgemma_pseudo_reports()
    direct_metrics = direct_metrics.copy()
    direct_summary = direct_summary.copy()
    direct_metrics.insert(0, "method", "direct_medgemma") if "method" not in direct_metrics.columns else None
    direct_summary.insert(0, "method", "direct_medgemma") if "method" not in direct_summary.columns else None
    return direct_metrics, direct_summary


def compare_gnn_rag_reports_to_direct():
    direct_metrics, direct_summary = load_or_evaluate_direct_medgemma_metrics()
    all_metrics = [direct_metrics]
    all_summaries = [direct_summary]
    paired_delta_rows = []

    for cfg, report_path in collect_gnn_rag_report_files():
        eval_frame = build_gnn_rag_eval_frame(cfg, report_path)
        if eval_frame.empty:
            print("No evaluable GNN-RAG reports for", cfg["config_name"])
            continue
        out_dir = GNN_RAG_EVAL_ROOT / cfg["config_name"]
        rag_metrics, rag_summary = evaluate_report_dataframe(eval_frame, "gnn_rag_report", cfg["config_name"], out_dir)
        all_metrics.append(rag_metrics)
        all_summaries.append(rag_summary)

        direct_subset = direct_metrics[direct_metrics["row_id"].isin(rag_metrics["row_id"])].copy()
        merged = rag_metrics.merge(direct_subset, on="row_id", suffixes=("_rag", "_direct"))
        for metric in ["bleu", "rouge_l", "cider", "bertscore_f1", "radgraph_f1", "clinical_label_micro_f1", "clinical_label_precision", "clinical_label_recall"]:
            rag_col = f"{metric}_rag"
            direct_col = f"{metric}_direct"
            if rag_col in merged.columns and direct_col in merged.columns:
                paired_delta_rows.append({
                    "method": cfg["config_name"],
                    "metric": metric,
                    "n_paired": int(merged[[rag_col, direct_col]].dropna().shape[0]),
                    "direct_mean": float(merged[direct_col].mean()),
                    "gnn_rag_mean": float(merged[rag_col].mean()),
                    "mean_delta_gnn_rag_minus_direct": float((merged[rag_col] - merged[direct_col]).mean()),
                })

    combined_metrics = pd.concat(all_metrics, ignore_index=True, sort=False)
    combined_summary = pd.concat(all_summaries, ignore_index=True, sort=False)
    paired_delta = pd.DataFrame(paired_delta_rows)

    combined_metrics_path = GNN_RAG_EVAL_ROOT / "direct_vs_gnn_rag_report_metrics_all.csv"
    combined_summary_path = GNN_RAG_EVAL_ROOT / "direct_vs_gnn_rag_report_metric_summary.csv"
    paired_delta_path = GNN_RAG_EVAL_ROOT / "direct_vs_gnn_rag_paired_metric_deltas.csv"
    combined_metrics.to_csv(combined_metrics_path, index=False)
    combined_summary.to_csv(combined_summary_path, index=False)
    paired_delta.to_csv(paired_delta_path, index=False)

    print("Saved combined metrics:", combined_metrics_path)
    print("Saved combined summary:", combined_summary_path)
    print("Saved paired deltas:", paired_delta_path)
    display(combined_summary)
    display(paired_delta)
    return combined_metrics, combined_summary, paired_delta


gnn_rag_report_metrics_df, gnn_rag_report_summary_df, gnn_rag_report_delta_df = compare_gnn_rag_reports_to_direct()


## Notes

This notebook does not train any GNN. The only change is inference-time retrieval: the query subgraph is selected using BiomedCLIP image embeddings plus BiomedCLIP embeddings of MedGemma pseudo reports. The fixed-threshold rows use threshold 0.5. The `val_tuned_f1` rows reuse thresholds saved during the original GNN training runs.